# Pipeline final Taller 2 v2 - version larga

Esta es la version autocontenida extendida del notebook de codigo. A
diferencia de la version modular, aca se embebe el codigo completo del
pipeline dentro del propio notebook: utilidades de datos, features
log-mel, metricas, modelo, entrenamiento curated e100, fine-tuning con
noisy y blend final.

Para que el notebook pueda ejecutarse en modo liviano sin depender del
kernel con Torch, las fuentes embebidas se escriben en
`codigo/work/_embedded_pipeline_src` y los pasos pesados se lanzan con
`.venv/bin/python` solamente cuando `RUN_FULL_PIPELINE=True`.


## Librerias y rutas


In [1]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "sample_submission.csv").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("No se encontro data/sample_submission.csv")

CODE_DIR = ROOT / "proyecto_actual_v2" / "codigo"
WORK_DIR = CODE_DIR / "work"
EMBEDDED_SRC = WORK_DIR / "_embedded_pipeline_src"
FIGURES_DIR = CODE_DIR / "figures"
DATA_DIR = ROOT / "data"
PYTHON_EXE = ROOT / ".venv" / "bin" / "python"
if not PYTHON_EXE.exists():
    PYTHON_EXE = Path(sys.executable)

WORK_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FINAL_SHA256 = "b29288caa6e7b37b29e830a29655decc7c6bc8110ca3a40b828f4dd2f5fabdcc"
FREESOUND2019_PRIVATE_LB = 0.68122
COURSE_PUBLIC_LB = 0.64307
PREVIOUS_3WAY_PRIVATE_LB = 0.67126


## Codigo fuente embebido

`EMBEDDED_FILES` contiene todos los archivos Python necesarios para
ejecutar el pipeline. Esta celda es larga a proposito: es la version
que permite entregar o revisar el flujo sin abrir `pipeline_src/`.


In [2]:
EMBEDDED_FILES = {
  "__init__.py": "\"\"\"Self-contained delivery pipeline helpers for Taller 2 v2.\"\"\"\n\n",
  "build_f1024_memmap_caches.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom build_logmel_image_cache import _extract_image  # noqa: E402\nfrom fat2019.data import load_training_labels  # noqa: E402\nfrom fat2019.submission import read_sample_submission  # noqa: E402\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Build f1024 clip-normalized memmap caches.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--force\", action=\"store_true\")\n    parser.add_argument(\"--sample-rate\", type=int, default=44100)\n    parser.add_argument(\"--n-mels\", type=int, default=128)\n    parser.add_argument(\"--frames\", type=int, default=1024)\n    parser.add_argument(\"--n-fft\", type=int, default=2048)\n    parser.add_argument(\"--hop-length\", type=int, default=512)\n    return parser\n\n\ndef memmap_cache_paths(data_dir: Path, *, split: str, n_mels: int, frames: int) -> tuple[Path, Path]:\n    stem = f\"{split}_logmel_image_m{n_mels}_f{frames}_x\"\n    return data_dir / f\"{stem}.npy\", data_dir / f\"{stem}_fnames.txt\"\n\n\ndef _split_fnames(data_dir: Path, split: str) -> list[str]:\n    if split == \"curated\":\n        return load_training_labels(data_dir / \"train_curated.csv\")[\"fname\"].astype(str).tolist()\n    if split == \"noisy\":\n        return load_training_labels(data_dir / \"train_noisy.csv\")[\"fname\"].astype(str).tolist()\n    if split == \"test\":\n        return read_sample_submission(data_dir / \"sample_submission.csv\")[\"fname\"].astype(str).tolist()\n    raise ValueError(f\"unknown split: {split}\")\n\n\ndef _build_split(\n    data_dir: Path,\n    split: str,\n    *,\n    sample_rate: int,\n    n_mels: int,\n    frames: int,\n    n_fft: int,\n    hop_length: int,\n    force: bool,\n) -> None:\n    x_path, fnames_path = memmap_cache_paths(data_dir, split=split, n_mels=n_mels, frames=frames)\n    fnames = _split_fnames(data_dir, split)\n    if x_path.exists() and fnames_path.exists() and not force:\n        existing = np.load(x_path, mmap_mode=\"r\")\n        if existing.shape == (len(fnames), n_mels, frames):\n            print(f\"skip existing memmap {x_path} {existing.shape}\")\n            return\n\n    import torchaudio\n\n    mel_transform = torchaudio.transforms.MelSpectrogram(\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        hop_length=hop_length,\n        n_mels=n_mels,\n        f_min=20.0,\n        f_max=sample_rate / 2,\n        power=2.0,\n        normalized=False,\n    )\n    amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype=\"power\")\n\n    x_path.parent.mkdir(parents=True, exist_ok=True)\n    images = np.lib.format.open_memmap(\n        x_path,\n        mode=\"w+\",\n        dtype=np.float16,\n        shape=(len(fnames), n_mels, frames),\n    )\n    for row_index, fname in enumerate(fnames, start=1):\n        images[row_index - 1] = _extract_image(\n            data_dir / fname,\n            mel_transform=mel_transform,\n            amplitude_to_db=amplitude_to_db,\n            sample_rate=sample_rate,\n            frames=frames,\n            normalization=\"clip\",\n            global_mean=None,\n            global_std=None,\n        )\n        if row_index % 250 == 0:\n            print(f\"{x_path.name}: {row_index}/{len(fnames)}\", flush=True)\n    images.flush()\n    del images\n    fnames_path.write_text(\"\\n\".join(fnames) + \"\\n\")\n    print(f\"wrote memmap {x_path} rows={len(fnames)}\")\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    for split in (\"curated\", \"noisy\", \"test\"):\n        _build_split(\n            args.data_dir,\n            split,\n            sample_rate=args.sample_rate,\n            n_mels=args.n_mels,\n            frames=args.frames,\n            n_fft=args.n_fft,\n            hop_length=args.hop_length,\n            force=args.force,\n        )\n    print(\"f1024_memmap_caches_ok\")\n\n\nif __name__ == \"__main__\":\n    main()\n\n",
  "build_logmel_image_cache.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom fat2019.data import load_training_labels\nfrom fat2019.spectrogram_images import (\n    crop_or_pad_frames,\n    normalize_logmel_image,\n    normalize_logmel_image_with_stats,\n)\nfrom fat2019.submission import read_sample_submission\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Build fixed-size log-mel image caches.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--splits\", default=\"curated,test\")\n    parser.add_argument(\"--sample-rate\", type=int, default=44100)\n    parser.add_argument(\"--n-mels\", type=int, default=128)\n    parser.add_argument(\"--frames\", type=int, default=512)\n    parser.add_argument(\"--n-fft\", type=int, default=2048)\n    parser.add_argument(\"--hop-length\", type=int, default=512)\n    parser.add_argument(\"--normalization\", choices=(\"clip\", \"global-mel\"), default=\"clip\")\n    parser.add_argument(\"--cache-tag\", default=None)\n    parser.add_argument(\"--force\", action=\"store_true\")\n    return parser\n\n\ndef parse_splits(raw_splits: str) -> list[str]:\n    splits = [split.strip() for split in raw_splits.split(\",\") if split.strip()]\n    valid_splits = {\"curated\", \"test\"}\n    unknown_splits = sorted(set(splits) - valid_splits)\n    if unknown_splits:\n        raise ValueError(f\"unknown cache split(s): {unknown_splits}\")\n    if not splits:\n        raise ValueError(\"expected at least one cache split\")\n    return splits\n\n\ndef logmel_image_cache_path(\n    data_dir: Path,\n    *,\n    split: str,\n    n_mels: int,\n    frames: int,\n    tag: str | None = None,\n) -> Path:\n    suffix = f\"_{tag}\" if tag else \"\"\n    return data_dir / f\"{split}_logmel_image_m{n_mels}_f{frames}{suffix}.npz\"\n\n\ndef _read_waveform(path: Path, *, sample_rate: int) -> Any:\n    # Third-party tensor type is intentionally lazy so unit tests do not require torch.\n    import torchaudio\n\n    waveform, original_sample_rate = torchaudio.load(str(path))\n    waveform = waveform.mean(dim=0, keepdim=True)\n    if original_sample_rate != sample_rate:\n        waveform = torchaudio.functional.resample(waveform, original_sample_rate, sample_rate)\n    return waveform\n\n\ndef _extract_logmel_image(\n    path: Path,\n    *,\n    mel_transform: Any,\n    amplitude_to_db: Any,\n    sample_rate: int,\n    frames: int,\n) -> np.ndarray:\n    import torch\n\n    waveform = _read_waveform(path, sample_rate=sample_rate)\n    with torch.no_grad():\n        mel = mel_transform(waveform)\n        logmel = amplitude_to_db(mel).squeeze(0).cpu().numpy()\n    return crop_or_pad_frames(logmel, frames=frames)\n\n\ndef _extract_image(\n    path: Path,\n    *,\n    mel_transform: Any,\n    amplitude_to_db: Any,\n    sample_rate: int,\n    frames: int,\n    normalization: str,\n    global_mean: np.ndarray | None,\n    global_std: np.ndarray | None,\n) -> np.ndarray:\n    image = _extract_logmel_image(\n        path,\n        mel_transform=mel_transform,\n        amplitude_to_db=amplitude_to_db,\n        sample_rate=sample_rate,\n        frames=frames,\n    )\n    if normalization == \"clip\":\n        normalized = normalize_logmel_image(image)\n    elif normalization == \"global-mel\":\n        if global_mean is None or global_std is None:\n            raise ValueError(\"global-mel normalization requires global mean and std\")\n        normalized = normalize_logmel_image_with_stats(image, mean=global_mean, std=global_std)\n    else:\n        raise ValueError(f\"unknown normalization: {normalization}\")\n    return normalized.astype(np.float16)\n\n\ndef compute_global_mel_stats(\n    data_dir: Path,\n    fnames: list[str],\n    *,\n    mel_transform: Any,\n    amplitude_to_db: Any,\n    sample_rate: int,\n    frames: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    total = None\n    total_squared = None\n    count = 0\n    for row_index, fname in enumerate(fnames, start=1):\n        image = _extract_logmel_image(\n            data_dir / fname,\n            mel_transform=mel_transform,\n            amplitude_to_db=amplitude_to_db,\n            sample_rate=sample_rate,\n            frames=frames,\n        ).astype(np.float64, copy=False)\n        band_sum = image.sum(axis=1)\n        band_squared_sum = np.square(image).sum(axis=1)\n        total = band_sum if total is None else total + band_sum\n        total_squared = (\n            band_squared_sum if total_squared is None else total_squared + band_squared_sum\n        )\n        count += image.shape[1]\n        if row_index % 250 == 0:\n            print(f\"global stats: {row_index}/{len(fnames)}\", flush=True)\n\n    if total is None or total_squared is None or count == 0:\n        raise ValueError(\"cannot compute global stats from an empty split\")\n\n    mean = total / count\n    variance = np.maximum(total_squared / count - np.square(mean), 1e-12)\n    return mean.astype(np.float32), np.sqrt(variance).astype(np.float32)\n\n\ndef build_cache(\n    data_dir: Path,\n    fnames: list[str],\n    *,\n    split: str,\n    sample_rate: int,\n    n_mels: int,\n    frames: int,\n    n_fft: int,\n    hop_length: int,\n    normalization: str,\n    cache_tag: str | None,\n    global_mean: np.ndarray | None,\n    global_std: np.ndarray | None,\n    force: bool,\n) -> None:\n    output_path = logmel_image_cache_path(\n        data_dir,\n        split=split,\n        n_mels=n_mels,\n        frames=frames,\n        tag=cache_tag,\n    )\n    if output_path.exists() and not force:\n        print(f\"skip existing {output_path}\")\n        return\n\n    import torchaudio\n\n    mel_transform = torchaudio.transforms.MelSpectrogram(\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        hop_length=hop_length,\n        n_mels=n_mels,\n        f_min=20.0,\n        f_max=sample_rate / 2,\n        power=2.0,\n        normalized=False,\n    )\n    amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype=\"power\")\n\n    images = np.empty((len(fnames), n_mels, frames), dtype=np.float16)\n    for row_index, fname in enumerate(fnames, start=1):\n        images[row_index - 1] = _extract_image(\n            data_dir / fname,\n            mel_transform=mel_transform,\n            amplitude_to_db=amplitude_to_db,\n            sample_rate=sample_rate,\n            frames=frames,\n            normalization=normalization,\n            global_mean=global_mean,\n            global_std=global_std,\n        )\n        if row_index % 250 == 0:\n            print(f\"{output_path.name}: {row_index}/{len(fnames)}\", flush=True)\n\n    metadata: dict[str, Any] = {\"normalization\": normalization}\n    if global_mean is not None and global_std is not None:\n        metadata[\"global_mean\"] = global_mean\n        metadata[\"global_std\"] = global_std\n    np.savez_compressed(output_path, x=images, fnames=np.array(fnames), **metadata)\n    print(f\"wrote {output_path} {images.shape}\")\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    splits = parse_splits(args.splits)\n    cache_tag = args.cache_tag\n    if cache_tag is None and args.normalization == \"global-mel\":\n        cache_tag = \"globalmel\"\n\n    global_mean = None\n    global_std = None\n    if args.normalization == \"global-mel\":\n        import torchaudio\n\n        mel_transform = torchaudio.transforms.MelSpectrogram(\n            sample_rate=args.sample_rate,\n            n_fft=args.n_fft,\n            hop_length=args.hop_length,\n            n_mels=args.n_mels,\n            f_min=20.0,\n            f_max=args.sample_rate / 2,\n            power=2.0,\n            normalized=False,\n        )\n        amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype=\"power\")\n        labels = load_training_labels(args.data_dir / \"train_curated.csv\")\n        global_mean, global_std = compute_global_mel_stats(\n            args.data_dir,\n            labels[\"fname\"].astype(str).tolist(),\n            mel_transform=mel_transform,\n            amplitude_to_db=amplitude_to_db,\n            sample_rate=args.sample_rate,\n            frames=args.frames,\n        )\n\n    if \"curated\" in splits:\n        labels = load_training_labels(args.data_dir / \"train_curated.csv\")\n        build_cache(\n            args.data_dir,\n            labels[\"fname\"].astype(str).tolist(),\n            split=\"curated\",\n            sample_rate=args.sample_rate,\n            n_mels=args.n_mels,\n            frames=args.frames,\n            n_fft=args.n_fft,\n            hop_length=args.hop_length,\n            normalization=args.normalization,\n            cache_tag=cache_tag,\n            global_mean=global_mean,\n            global_std=global_std,\n            force=args.force,\n        )\n    if \"test\" in splits:\n        sample = read_sample_submission(args.data_dir / \"sample_submission.csv\")\n        build_cache(\n            args.data_dir,\n            sample[\"fname\"].astype(str).tolist(),\n            split=\"test\",\n            sample_rate=args.sample_rate,\n            n_mels=args.n_mels,\n            frames=args.frames,\n            n_fft=args.n_fft,\n            hop_length=args.hop_length,\n            normalization=args.normalization,\n            cache_tag=cache_tag,\n            global_mean=global_mean,\n            global_std=global_std,\n            force=args.force,\n        )\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "build_noisy_caches.py": "from __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom build_logmel_image_cache import (  # noqa: E402\n    build_cache,\n    compute_global_mel_stats,\n    logmel_image_cache_path,\n)\nfrom fat2019.data import load_training_labels  # noqa: E402\nfrom final_config import BRANCHES  # noqa: E402\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Build train_noisy log-mel image caches for the final pipeline.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--force\", action=\"store_true\")\n    parser.add_argument(\"--sample-rate\", type=int, default=44100)\n    parser.add_argument(\"--n-fft\", type=int, default=2048)\n    parser.add_argument(\"--hop-length\", type=int, default=512)\n    return parser\n\n\ndef _global_stats_from_curated_cache(\n    data_dir: Path,\n    *,\n    n_mels: int,\n    frames: int,\n) -> tuple[np.ndarray | None, np.ndarray | None]:\n    curated_cache_path = logmel_image_cache_path(\n        data_dir,\n        split=\"curated\",\n        n_mels=n_mels,\n        frames=frames,\n        tag=\"globalmel\",\n    )\n    if not curated_cache_path.exists():\n        return None, None\n    cache = np.load(curated_cache_path, allow_pickle=False)\n    if \"global_mean\" not in cache or \"global_std\" not in cache:\n        return None, None\n    return cache[\"global_mean\"].astype(np.float32), cache[\"global_std\"].astype(np.float32)\n\n\ndef _compute_global_stats(\n    data_dir: Path,\n    *,\n    sample_rate: int,\n    n_mels: int,\n    frames: int,\n    n_fft: int,\n    hop_length: int,\n) -> tuple[np.ndarray, np.ndarray]:\n    import torchaudio\n\n    labels = load_training_labels(data_dir / \"train_curated.csv\")\n    mel_transform = torchaudio.transforms.MelSpectrogram(\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        hop_length=hop_length,\n        n_mels=n_mels,\n        f_min=20.0,\n        f_max=sample_rate / 2,\n        power=2.0,\n        normalized=False,\n    )\n    amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype=\"power\")\n    return compute_global_mel_stats(\n        data_dir,\n        labels[\"fname\"].astype(str).tolist(),\n        mel_transform=mel_transform,\n        amplitude_to_db=amplitude_to_db,\n        sample_rate=sample_rate,\n        frames=frames,\n    )\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    data_dir = args.data_dir\n    noisy_labels = load_training_labels(data_dir / \"train_noisy.csv\")\n    noisy_fnames = noisy_labels[\"fname\"].astype(str).tolist()\n    unique_configs = sorted(\n        {(branch.n_mels, branch.frames, branch.cache_tag) for branch in BRANCHES},\n        key=lambda item: (item[0], item[1], item[2] or \"\"),\n    )\n\n    for n_mels, frames, cache_tag in unique_configs:\n        if frames == 1024 and cache_tag is None:\n            print(\"skip noisy npz cache for f1024; memmap builder handles it\")\n            continue\n\n        normalization = \"global-mel\" if cache_tag == \"globalmel\" else \"clip\"\n        global_mean = None\n        global_std = None\n        if normalization == \"global-mel\":\n            global_mean, global_std = _global_stats_from_curated_cache(\n                data_dir,\n                n_mels=n_mels,\n                frames=frames,\n            )\n            if global_mean is None or global_std is None:\n                global_mean, global_std = _compute_global_stats(\n                    data_dir,\n                    sample_rate=args.sample_rate,\n                    n_mels=n_mels,\n                    frames=frames,\n                    n_fft=args.n_fft,\n                    hop_length=args.hop_length,\n                )\n\n        output_path = logmel_image_cache_path(\n            data_dir,\n            split=\"noisy\",\n            n_mels=n_mels,\n            frames=frames,\n            tag=cache_tag,\n        )\n        build_cache(\n            data_dir,\n            noisy_fnames,\n            split=\"noisy\",\n            sample_rate=args.sample_rate,\n            n_mels=n_mels,\n            frames=frames,\n            n_fft=args.n_fft,\n            hop_length=args.hop_length,\n            normalization=normalization,\n            cache_tag=cache_tag,\n            global_mean=global_mean,\n            global_std=global_std,\n            force=args.force,\n        )\n        cache = np.load(output_path, allow_pickle=False)\n        if len(cache[\"x\"]) != len(noisy_fnames):\n            raise ValueError(f\"{output_path} row mismatch: {len(cache['x'])} vs {len(noisy_fnames)}\")\n        print(f\"cache_ok {output_path} rows={len(cache['x'])}\")\n\n    print(\"noisy_caches_ok\")\n\n\nif __name__ == \"__main__\":\n    main()\n\n",
  "evaluate_and_blend.py": "from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom fat2019.metrics import calculate_overall_lwlrap  # noqa: E402\nfrom fat2019.submission import (  # noqa: E402\n    blend_submissions,\n    label_columns_from_sample,\n    read_sample_submission,\n    validate_submission,\n    write_submission,\n)\nfrom final_config import BRANCHES  # noqa: E402\n\n\nCOMPETITION = \"freesound-audio-tagging-2019\"\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Blend noisy fine-tuned branches and summarize metrics.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--work-dir\", type=Path, default=Path(\"proyecto_actual_v2/codigo/work\"))\n    parser.add_argument(\"--output-dir\", type=Path, default=Path(\"proyecto_actual_v2/codigo\"))\n    return parser\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _load_branch_metadata(work_dir: Path, branch_name: str) -> dict[str, object]:\n    path = work_dir / \"runs\" / branch_name / \"experiments\" / \"metadata.json\"\n    if not path.exists():\n        raise FileNotFoundError(f\"missing branch metadata: {path}\")\n    return json.loads(path.read_text())\n\n\ndef _blend_validation_scores(work_dir: Path) -> tuple[float, list[dict[str, object]]]:\n    weighted_scores: np.ndarray | None = None\n    targets: np.ndarray | None = None\n    branch_rows: list[dict[str, object]] = []\n    for branch in BRANCHES:\n        experiments_dir = branch.run_dir(work_dir) / \"experiments\"\n        scores = np.load(experiments_dir / \"valid_scores_final.npy\")\n        branch_targets = np.load(experiments_dir / \"valid_targets.npy\")\n        if targets is None:\n            targets = branch_targets\n        elif targets.shape != branch_targets.shape or not np.array_equal(targets, branch_targets):\n            raise ValueError(f\"validation targets differ for branch {branch.name}\")\n        if weighted_scores is None:\n            weighted_scores = np.zeros_like(scores, dtype=np.float64)\n        weighted_scores += branch.ensemble_weight * scores.astype(np.float64)\n        metadata = _load_branch_metadata(work_dir, branch.name)\n        branch_rows.append(\n            {\n                \"branch\": branch.name,\n                \"weight\": branch.ensemble_weight,\n                \"baseline_lwlrap\": metadata[\"baseline_lwlrap\"],\n                \"best_lwlrap\": metadata[\"best_lwlrap\"],\n                \"final_lwlrap\": metadata[\"final_lwlrap\"],\n                \"best_epoch\": metadata[\"best_epoch\"],\n                \"submission_checkpoint\": metadata.get(\"submission_checkpoint\", \"unknown\"),\n            }\n        )\n    if weighted_scores is None or targets is None:\n        raise ValueError(\"no branch validation scores found\")\n    local_lwlrap = calculate_overall_lwlrap(targets, weighted_scores)\n    return float(local_lwlrap), branch_rows\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    sample = read_sample_submission(args.data_dir / \"sample_submission.csv\")\n    label_columns = label_columns_from_sample(sample)\n\n    submissions = []\n    for branch in BRANCHES:\n        path = branch.run_dir(args.work_dir) / \"submissions\" / \"small_logmel_cnn.csv\"\n        if not path.exists():\n            raise FileNotFoundError(f\"missing branch submission: {path}\")\n        submission = pd.read_csv(path)\n        validate_submission(submission, label_columns, expected_rows=len(sample))\n        submissions.append(submission)\n\n    ensemble_dir = args.work_dir / \"runs\" / \"ensemble\"\n    ensemble_dir.mkdir(parents=True, exist_ok=True)\n    blended = blend_submissions(\n        submissions,\n        weights=[branch.ensemble_weight for branch in BRANCHES],\n        label_columns=label_columns,\n    )\n    ensemble_submission_path = ensemble_dir / \"submission.csv\"\n    write_submission(blended, ensemble_submission_path, label_columns)\n    submission_hash = sha256(ensemble_submission_path)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    delivery_submission_path = args.output_dir / \"submission.csv\"\n    write_submission(blended, delivery_submission_path, label_columns)\n\n    local_lwlrap, branch_rows = _blend_validation_scores(args.work_dir)\n\n    results_path = args.output_dir / \"pipeline_final_taller_2_v2_results.md\"\n    lines = [\n        \"# Noisy fine-tune results\",\n        \"\",\n        f\"Generated: {datetime.now(timezone.utc).isoformat()}\",\n        \"\",\n        \"## Local validation\",\n        \"\",\n        \"| Branch | Weight | Baseline lwlrap | Best lwlrap | Final lwlrap | Best epoch | Submission ckpt |\",\n        \"|---|---:|---:|---:|---:|---:|---|\",\n    ]\n    for row in branch_rows:\n        lines.append(\n            \"| {branch} | {weight:.3f} | {baseline_lwlrap:.6f} | {best_lwlrap:.6f} | {final_lwlrap:.6f} | {best_epoch} | {submission_checkpoint} |\".format(\n                **row\n            )\n        )\n    lines.extend(\n        [\n            \"\",\n            f\"Ensemble validation lwlrap: `{local_lwlrap:.6f}`\",\n            \"\",\n            \"## Submission\",\n            \"\",\n            f\"- Path: `{ensemble_submission_path}`\",\n            f\"- Delivery path: `{delivery_submission_path}`\",\n            f\"- SHA256: `{submission_hash}`\",\n            f\"- Rows: `{len(blended)}`\",\n            f\"- Labels: `{len(label_columns)}`\",\n            \"\",\n            \"Kaggle is intentionally not submitted from this delivery script.\",\n            \"\",\n        ]\n    )\n    results_path.write_text(\"\\n\".join(lines))\n    print(f\"wrote {ensemble_submission_path}\")\n    print(f\"wrote {results_path}\")\n    print(f\"ensemble_valid_lwlrap={local_lwlrap:.6f}\")\n    print(f\"submission_sha256={submission_hash}\")\n    print(\"evaluate_and_blend_ok\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "fat2019/__init__.py": "\"\"\"Utilities for Freesound Audio Tagging 2019 experiments.\"\"\"\n",
  "fat2019/data.py": "from __future__ import annotations\n\nfrom collections import Counter\nfrom pathlib import Path\n\nimport pandas as pd\n\n\nCORRUPT_OR_BAD_LABEL_FILES = {\n    \"f76181c4.wav\",\n    \"77b925c2.wav\",\n    \"6a1f682a.wav\",\n    \"c7db12aa.wav\",\n    \"7752cc8a.wav\",\n    \"1d44b0bd.wav\",\n}\n\n\ndef split_labels(labels: str) -> list[str]:\n    if not isinstance(labels, str) or not labels.strip():\n        return []\n    return [label.strip() for label in labels.split(\",\") if label.strip()]\n\n\ndef load_training_labels(path: Path, *, drop_known_bad: bool = True) -> pd.DataFrame:\n    labels = pd.read_csv(path)\n    required_columns = {\"fname\", \"labels\"}\n    missing = required_columns - set(labels.columns)\n    if missing:\n        raise ValueError(f\"{path} is missing required columns: {sorted(missing)}\")\n    if drop_known_bad:\n        labels = labels[~labels[\"fname\"].isin(CORRUPT_OR_BAD_LABEL_FILES)].copy()\n    return labels\n\n\ndef class_priors_from_training_labels(labels: pd.DataFrame) -> dict[str, float]:\n    counts: Counter[str] = Counter()\n    for row_labels in labels[\"labels\"]:\n        counts.update(split_labels(row_labels))\n\n    total_rows = len(labels)\n    if total_rows == 0:\n        raise ValueError(\"cannot calculate priors from an empty labels dataframe\")\n    return {label: count / total_rows for label, count in counts.items()}\n",
  "fat2019/features.py": "from __future__ import annotations\n\nimport numpy as np\nfrom scipy import signal\nfrom scipy.io import wavfile\n\n\ndef _hz_to_mel(frequency_hz: np.ndarray | float) -> np.ndarray | float:\n    return 2595.0 * np.log10(1.0 + np.asarray(frequency_hz) / 700.0)\n\n\ndef _mel_to_hz(mels: np.ndarray) -> np.ndarray:\n    return 700.0 * (10.0 ** (mels / 2595.0) - 1.0)\n\n\ndef mel_filterbank(\n    *,\n    sample_rate: int,\n    n_fft: int,\n    n_mels: int,\n    fmin: float = 0.0,\n    fmax: float | None = None,\n) -> np.ndarray:\n    if fmax is None:\n        fmax = sample_rate / 2\n    if not 0 <= fmin < fmax <= sample_rate / 2:\n        raise ValueError(\"expected 0 <= fmin < fmax <= sample_rate / 2\")\n\n    mel_points = np.linspace(_hz_to_mel(fmin), _hz_to_mel(fmax), n_mels + 2)\n    hz_points = _mel_to_hz(mel_points)\n    fft_frequencies = np.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)\n\n    filters = np.zeros((len(fft_frequencies), n_mels), dtype=np.float32)\n    for mel_index in range(n_mels):\n        lower = hz_points[mel_index]\n        center = hz_points[mel_index + 1]\n        upper = hz_points[mel_index + 2]\n\n        left_slope = (fft_frequencies - lower) / (center - lower)\n        right_slope = (upper - fft_frequencies) / (upper - center)\n        filters[:, mel_index] = np.maximum(0.0, np.minimum(left_slope, right_slope))\n\n    return filters\n\n\ndef log_mel_spectrogram(\n    waveform: np.ndarray,\n    *,\n    sample_rate: int,\n    n_fft: int = 1024,\n    hop_length: int = 512,\n    n_mels: int = 80,\n    fmin: float = 0.0,\n    fmax: float | None = 4000.0,\n    eps: float = 1e-6,\n) -> np.ndarray:\n    waveform = np.asarray(waveform, dtype=np.float32)\n    if waveform.ndim != 1:\n        raise ValueError(f\"waveform must be mono with shape (samples,), got {waveform.shape}\")\n    if waveform.size == 0:\n        raise ValueError(\"waveform must not be empty\")\n\n    _, _, stft = signal.stft(\n        waveform,\n        fs=sample_rate,\n        window=\"hann\",\n        nperseg=n_fft,\n        noverlap=n_fft - hop_length,\n        nfft=n_fft,\n        boundary=None,\n        padded=False,\n    )\n    magnitude = np.abs(stft).astype(np.float32)\n    filters = mel_filterbank(\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        n_mels=n_mels,\n        fmin=fmin,\n        fmax=fmax,\n    )\n    mel = filters.T @ magnitude\n    return np.log(mel + eps).astype(np.float32)\n\n\ndef extract_log_mel_stats(\n    waveform: np.ndarray,\n    *,\n    sample_rate: int,\n    n_fft: int = 1024,\n    hop_length: int = 512,\n    n_mels: int = 80,\n    fmin: float = 20.0,\n    fmax: float | None = None,\n) -> np.ndarray:\n    log_mel = log_mel_spectrogram(\n        waveform,\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        hop_length=hop_length,\n        n_mels=n_mels,\n        fmin=fmin,\n        fmax=fmax,\n    )\n    stats = [\n        np.mean(log_mel, axis=1),\n        np.std(log_mel, axis=1),\n        np.max(log_mel, axis=1),\n        np.percentile(log_mel, 75, axis=1),\n    ]\n    waveform_float = np.asarray(waveform, dtype=np.float32)\n    duration = np.array([waveform_float.size / sample_rate], dtype=np.float32)\n    rms = np.array([np.sqrt(np.mean(np.square(waveform_float)))], dtype=np.float32)\n    zero_crossing_rate = np.array(\n        [np.mean(np.diff(np.signbit(waveform_float)).astype(np.float32))],\n        dtype=np.float32,\n    )\n    return np.concatenate([*stats, duration, rms, zero_crossing_rate]).astype(np.float32)\n\n\ndef extract_log_mel_stats_extended(\n    waveform: np.ndarray,\n    *,\n    sample_rate: int,\n    n_fft: int = 1024,\n    hop_length: int = 512,\n    n_mels: int = 80,\n    fmin: float = 20.0,\n    fmax: float | None = None,\n) -> np.ndarray:\n    log_mel = log_mel_spectrogram(\n        waveform,\n        sample_rate=sample_rate,\n        n_fft=n_fft,\n        hop_length=hop_length,\n        n_mels=n_mels,\n        fmin=fmin,\n        fmax=fmax,\n    )\n    stats = [\n        np.mean(log_mel, axis=1),\n        np.std(log_mel, axis=1),\n        np.min(log_mel, axis=1),\n        np.percentile(log_mel, 25, axis=1),\n        np.percentile(log_mel, 50, axis=1),\n        np.percentile(log_mel, 75, axis=1),\n        np.max(log_mel, axis=1),\n    ]\n    waveform_float = np.asarray(waveform, dtype=np.float32)\n    duration = np.array([waveform_float.size / sample_rate], dtype=np.float32)\n    rms = np.array([np.sqrt(np.mean(np.square(waveform_float)))], dtype=np.float32)\n    zero_crossing_rate = np.array(\n        [np.mean(np.diff(np.signbit(waveform_float)).astype(np.float32))],\n        dtype=np.float32,\n    )\n    return np.concatenate([*stats, duration, rms, zero_crossing_rate]).astype(np.float32)\n\n\ndef read_wav_mono(path: str) -> tuple[int, np.ndarray]:\n    sample_rate, waveform = wavfile.read(path)\n    waveform = np.asarray(waveform)\n    if waveform.ndim == 2:\n        waveform = np.mean(waveform, axis=1)\n    if np.issubdtype(waveform.dtype, np.integer):\n        max_value = np.iinfo(waveform.dtype).max\n        waveform = waveform.astype(np.float32) / max_value\n    else:\n        waveform = waveform.astype(np.float32)\n    return int(sample_rate), waveform\n",
  "fat2019/labels.py": "from __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom fat2019.data import split_labels\n\n\ndef dataframe_to_multihot(labels: pd.DataFrame, label_columns: list[str]) -> np.ndarray:\n    label_to_index = {label: index for index, label in enumerate(label_columns)}\n    y = np.zeros((len(labels), len(label_columns)), dtype=np.float32)\n    for row_index, row_labels in enumerate(labels[\"labels\"]):\n        for label in split_labels(row_labels):\n            if label in label_to_index:\n                y[row_index, label_to_index[label]] = 1.0\n    return y\n",
  "fat2019/metrics.py": "from __future__ import annotations\n\nimport numpy as np\n\n\ndef _one_sample_positive_class_precisions(\n    scores: np.ndarray,\n    truth: np.ndarray,\n) -> tuple[np.ndarray, np.ndarray]:\n    positive_class_indices = np.flatnonzero(truth > 0)\n    if len(positive_class_indices) == 0:\n        return positive_class_indices, np.zeros(0, dtype=np.float64)\n\n    num_classes = scores.shape[0]\n    retrieved_classes = np.argsort(scores)[::-1]\n    class_rankings = np.zeros(num_classes, dtype=np.int32)\n    class_rankings[retrieved_classes] = np.arange(num_classes)\n\n    retrieved_class_true = np.zeros(num_classes, dtype=np.bool_)\n    retrieved_class_true[class_rankings[positive_class_indices]] = True\n    retrieved_cumulative_hits = np.cumsum(retrieved_class_true)\n\n    precision_at_hits = (\n        retrieved_cumulative_hits[class_rankings[positive_class_indices]]\n        / (1 + class_rankings[positive_class_indices].astype(np.float64))\n    )\n    return positive_class_indices, precision_at_hits\n\n\ndef calculate_per_class_lwlrap(\n    truth: np.ndarray,\n    scores: np.ndarray,\n) -> tuple[np.ndarray, np.ndarray]:\n    if truth.shape != scores.shape:\n        raise ValueError(\n            f\"truth and scores must have the same shape, got {truth.shape} and {scores.shape}\"\n        )\n\n    num_samples, num_classes = scores.shape\n    precisions_for_samples_by_classes = np.zeros(\n        (num_samples, num_classes),\n        dtype=np.float64,\n    )\n\n    for sample_index in range(num_samples):\n        positive_class_indices, precision_at_hits = _one_sample_positive_class_precisions(\n            scores[sample_index, :],\n            truth[sample_index, :],\n        )\n        precisions_for_samples_by_classes[sample_index, positive_class_indices] = (\n            precision_at_hits\n        )\n\n    labels_per_class = np.sum(truth > 0, axis=0)\n    total_labels = np.sum(labels_per_class)\n    if total_labels == 0:\n        raise ValueError(\"truth must contain at least one positive label\")\n\n    weight_per_class = labels_per_class / float(total_labels)\n    per_class_lwlrap = np.sum(precisions_for_samples_by_classes, axis=0) / np.maximum(\n        1,\n        labels_per_class,\n    )\n    return per_class_lwlrap, weight_per_class\n\n\ndef calculate_overall_lwlrap(truth: np.ndarray, scores: np.ndarray) -> float:\n    per_class_lwlrap, weight_per_class = calculate_per_class_lwlrap(truth, scores)\n    return float(np.sum(per_class_lwlrap * weight_per_class))\n",
  "fat2019/neural_helpers.py": "from __future__ import annotations\n\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\n\n\ndef compute_pos_weight(targets: np.ndarray) -> np.ndarray:\n    if targets.ndim != 2:\n        raise ValueError(f\"expected 2D targets, got shape {targets.shape}\")\n\n    positives = targets.sum(axis=0).astype(np.float32)\n    negatives = (targets.shape[0] - positives).astype(np.float32)\n    weights = np.ones_like(positives, dtype=np.float32)\n    non_empty = positives > 0.0\n    weights[non_empty] = negatives[non_empty] / positives[non_empty]\n    return np.clip(weights, 1.0, 20.0).astype(np.float32)\n\n\ndef sigmoid_numpy(logits: np.ndarray) -> np.ndarray:\n    clipped = np.clip(logits.astype(np.float64, copy=False), -80.0, 80.0)\n    return (1.0 / (1.0 + np.exp(-clipped))).astype(np.float32)\n\n\ndef make_train_valid_indices(\n    *,\n    num_rows: int,\n    test_size: float,\n    seed: int,\n    full_train: bool,\n) -> tuple[np.ndarray, np.ndarray]:\n    if num_rows <= 0:\n        raise ValueError(\"num_rows must be positive\")\n    indices = np.arange(num_rows)\n    if full_train:\n        return indices, np.array([], dtype=np.int64)\n    train_indices, valid_indices = train_test_split(\n        indices,\n        test_size=test_size,\n        random_state=seed,\n    )\n    return train_indices, valid_indices\n",
  "fat2019/spectrogram_images.py": "from __future__ import annotations\n\nimport numpy as np\n\n\ndef crop_or_pad_frames(image: np.ndarray, *, frames: int) -> np.ndarray:\n    if image.ndim != 2:\n        raise ValueError(f\"expected 2D image, got shape {image.shape}\")\n    if frames <= 0:\n        raise ValueError(\"frames must be positive\")\n\n    current_frames = image.shape[1]\n    if current_frames == frames:\n        return image.astype(np.float32, copy=False)\n    if current_frames > frames:\n        start = (current_frames - frames) // 2\n        return image[:, start : start + frames].astype(np.float32, copy=False)\n\n    output = np.zeros((image.shape[0], frames), dtype=np.float32)\n    output[:, :current_frames] = image.astype(np.float32, copy=False)\n    return output\n\n\ndef normalize_logmel_image(image: np.ndarray) -> np.ndarray:\n    normalized = image.astype(np.float32, copy=False)\n    mean = float(normalized.mean())\n    std = float(normalized.std())\n    return (normalized - mean) / max(std, 1e-6)\n\n\ndef normalize_logmel_image_with_stats(\n    image: np.ndarray,\n    *,\n    mean: np.ndarray,\n    std: np.ndarray,\n) -> np.ndarray:\n    if image.ndim != 2:\n        raise ValueError(f\"expected 2D image, got shape {image.shape}\")\n    mean = np.asarray(mean, dtype=np.float32)\n    std = np.asarray(std, dtype=np.float32)\n    if mean.shape != (image.shape[0],):\n        raise ValueError(f\"mean must have shape ({image.shape[0]},), got {mean.shape}\")\n    if std.shape != (image.shape[0],):\n        raise ValueError(f\"std must have shape ({image.shape[0]},), got {std.shape}\")\n\n    safe_std = np.maximum(std, 1e-6)\n    return ((image.astype(np.float32, copy=False) - mean[:, None]) / safe_std[:, None]).astype(\n        np.float32,\n        copy=False,\n    )\n",
  "fat2019/submission.py": "from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Protocol\n\nimport numpy as np\nimport pandas as pd\n\n\nclass ProbabilityModel(Protocol):\n    def predict_proba(self, x: np.ndarray) -> np.ndarray:\n        \"\"\"Predict class probabilities for feature rows.\"\"\"\n\n\ndef label_columns_from_sample(sample_submission: pd.DataFrame) -> list[str]:\n    if \"fname\" not in sample_submission.columns:\n        raise ValueError(\"sample submission must include fname\")\n    return [column for column in sample_submission.columns if column != \"fname\"]\n\n\ndef validate_submission(\n    submission: pd.DataFrame,\n    label_columns: list[str],\n    *,\n    expected_rows: int | None = None,\n) -> None:\n    expected_columns = [\"fname\", *label_columns]\n    if list(submission.columns) != expected_columns:\n        raise ValueError(\n            \"submission columns do not match expected columns/order: \"\n            f\"expected {expected_columns[:4]}..., got {list(submission.columns)[:4]}...\"\n        )\n    if expected_rows is not None and len(submission) != expected_rows:\n        raise ValueError(f\"expected {expected_rows} rows, got {len(submission)}\")\n    if submission[\"fname\"].isna().any():\n        raise ValueError(\"submission contains empty fname values\")\n\n    probabilities = submission[label_columns]\n    if probabilities.isna().any().any():\n        raise ValueError(\"submission probabilities contain NaN values\")\n    if ((probabilities < 0.0) | (probabilities > 1.0)).any().any():\n        raise ValueError(\"submission probabilities must be in [0, 1]\")\n\n\ndef build_prior_submission(\n    sample_submission: pd.DataFrame,\n    class_priors: dict[str, float],\n) -> pd.DataFrame:\n    label_columns = label_columns_from_sample(sample_submission)\n    submission = pd.DataFrame({\"fname\": sample_submission[\"fname\"].astype(str)})\n    for label in label_columns:\n        submission[label] = float(class_priors.get(label, 0.0))\n    validate_submission(submission, label_columns, expected_rows=len(sample_submission))\n    return submission\n\n\ndef build_model_submission(\n    sample_submission: pd.DataFrame,\n    label_columns: list[str],\n    model: ProbabilityModel,\n    features: np.ndarray,\n) -> pd.DataFrame:\n    scores = np.asarray(model.predict_proba(features), dtype=np.float64)\n    expected_shape = (len(sample_submission), len(label_columns))\n    if scores.shape != expected_shape:\n        raise ValueError(f\"model scores must have shape {expected_shape}, got {scores.shape}\")\n\n    scores = np.clip(scores, 0.0, 1.0)\n    submission = pd.DataFrame(scores, columns=label_columns)\n    submission.insert(0, \"fname\", sample_submission[\"fname\"].astype(str).to_numpy())\n    validate_submission(submission, label_columns, expected_rows=len(sample_submission))\n    return submission\n\n\ndef blend_submissions(\n    submissions: list[pd.DataFrame],\n    *,\n    weights: list[float],\n    label_columns: list[str],\n) -> pd.DataFrame:\n    if not submissions:\n        raise ValueError(\"expected at least one submission\")\n    if len(submissions) != len(weights):\n        raise ValueError(\"submissions and weights must have the same length\")\n    if any(weight < 0.0 for weight in weights):\n        raise ValueError(\"weights must be non-negative\")\n    total_weight = float(sum(weights))\n    if total_weight <= 0.0:\n        raise ValueError(\"weights must sum to a positive value\")\n\n    expected_rows = len(submissions[0])\n    for submission in submissions:\n        validate_submission(submission, label_columns, expected_rows=expected_rows)\n        if not submission[\"fname\"].equals(submissions[0][\"fname\"]):\n            raise ValueError(\"all submissions must have the same fname order\")\n\n    normalized_weights = np.asarray(weights, dtype=np.float64) / total_weight\n    scores = np.zeros((expected_rows, len(label_columns)), dtype=np.float64)\n    for submission, weight in zip(submissions, normalized_weights):\n        scores += weight * submission[label_columns].to_numpy(dtype=np.float64)\n\n    blended = pd.DataFrame(np.clip(scores, 0.0, 1.0), columns=label_columns)\n    blended.insert(0, \"fname\", submissions[0][\"fname\"].astype(str).to_numpy())\n    validate_submission(blended, label_columns, expected_rows=expected_rows)\n    return blended\n\n\ndef read_sample_submission(path: Path) -> pd.DataFrame:\n    return pd.read_csv(path)\n\n\ndef write_submission(\n    submission: pd.DataFrame,\n    output_path: Path,\n    label_columns: list[str],\n) -> None:\n    validate_submission(submission, label_columns)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    submission.to_csv(output_path, index=False)\n",
  "final_config.py": "from __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass BranchConfig:\n    name: str\n    source_run: str\n    ensemble_weight: float\n    n_mels: int\n    frames: int\n    cache_tag: str | None\n    batch_size: int\n    base_epochs: int\n    base_lr: float\n    base_weight_decay: float\n    architecture: str\n    activation: str\n    initializer: str\n    optimizer: str\n    scheduler: str\n    lr_milestones: tuple[int, ...]\n    head_hidden: int\n    head_dropout: float\n    block_dropout: float = 0.0\n    time_reverse_probability: float = 0.0\n    contrast_strength: float = 0.0\n\n    def source_checkpoint(self, work_dir: Path) -> Path:\n        return work_dir / \"models\" / self.source_run / \"small_logmel_cnn_best.pt\"\n\n    def source_metadata(self, work_dir: Path) -> Path:\n        return work_dir / \"experiments\" / self.source_run / \"small_logmel_cnn_metadata.json\"\n\n    def run_dir(self, work_dir: Path) -> Path:\n        return work_dir / \"runs\" / self.name\n\n\nBRANCHES: tuple[BranchConfig, ...] = (\n    BranchConfig(\n        name=\"separable_headsep\",\n        source_run=\"separable_headsep_e100_seed42\",\n        ensemble_weight=0.25,\n        n_mels=128,\n        frames=512,\n        cache_tag=None,\n        batch_size=24,\n        base_epochs=100,\n        base_lr=1e-3,\n        base_weight_decay=1e-4,\n        architecture=\"separable_residual\",\n        activation=\"relu\",\n        initializer=\"he_normal\",\n        optimizer=\"adam\",\n        scheduler=\"multistep\",\n        lr_milestones=(27, 36, 43, 49, 52),\n        head_hidden=256,\n        head_dropout=0.30,\n    ),\n    BranchConfig(\n        name=\"globalmel_sep_temporal\",\n        source_run=\"globalmel_sep_temporal_e100_seed42\",\n        ensemble_weight=0.375,\n        n_mels=128,\n        frames=512,\n        cache_tag=\"globalmel\",\n        batch_size=24,\n        base_epochs=100,\n        base_lr=1e-3,\n        base_weight_decay=1e-4,\n        architecture=\"separable_temporal_bigru\",\n        activation=\"silu\",\n        initializer=\"he_normal\",\n        optimizer=\"adamw\",\n        scheduler=\"multistep\",\n        lr_milestones=(25, 39),\n        head_hidden=0,\n        head_dropout=0.30,\n    ),\n    BranchConfig(\n        name=\"sep_temporal_f1024\",\n        source_run=\"sep_temporal_f1024_e100_seed42\",\n        ensemble_weight=0.375,\n        n_mels=128,\n        frames=1024,\n        cache_tag=None,\n        batch_size=12,\n        base_epochs=100,\n        base_lr=1e-3,\n        base_weight_decay=1e-4,\n        architecture=\"separable_temporal_bigru\",\n        activation=\"silu\",\n        initializer=\"he_normal\",\n        optimizer=\"adamw\",\n        scheduler=\"multistep\",\n        lr_milestones=(19, 25),\n        head_hidden=0,\n        head_dropout=0.30,\n    ),\n)\n\n\n@dataclass(frozen=True)\nclass FineTuneConfig:\n    epochs: int = 30\n    lr: float = 1e-4\n    min_lr: float = 1e-6\n    noisy_loss_weight: float = 0.30\n    curated_loss_weight: float = 1.00\n    validation_test_size: float = 0.20\n    seed: int = 42\n    gaussian_noise_std: float = 0.015\n    num_workers: int = 2\n    submission_checkpoint: str = \"final\"\n    augmentations: tuple[str, ...] = field(\n        default=(\"time shift\", \"time mask\", \"frequency mask\", \"gaussian noise\")\n    )\n\n\nFINE_TUNE = FineTuneConfig()\n\n",
  "finetune_mixed_noisy.py": "from __future__ import annotations\n\nimport argparse\nimport gc\nimport json\nimport math\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset, Sampler\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom build_logmel_image_cache import logmel_image_cache_path  # noqa: E402\nfrom fat2019.data import load_training_labels  # noqa: E402\nfrom fat2019.labels import dataframe_to_multihot  # noqa: E402\nfrom fat2019.metrics import calculate_overall_lwlrap  # noqa: E402\nfrom fat2019.neural_helpers import (  # noqa: E402\n    compute_pos_weight,\n    make_train_valid_indices,\n    sigmoid_numpy,\n)\nfrom fat2019.submission import (  # noqa: E402\n    label_columns_from_sample,\n    read_sample_submission,\n    write_submission,\n)\nfrom final_config import BRANCHES, FINE_TUNE, BranchConfig  # noqa: E402\nfrom train_logmel_cnn import (  # noqa: E402\n    LogmelDataset,\n    SmallLogmelCnn,\n    build_optimizer,\n    build_scheduler,\n)\n\n\n@dataclass(frozen=True)\nclass BranchRunResult:\n    branch: str\n    baseline_lwlrap: float\n    best_lwlrap: float\n    final_lwlrap: float\n    best_epoch: int\n    submission_path: Path\n    best_model_path: Path\n    final_model_path: Path\n\n\nclass IndexedLogmelDataset(Dataset):\n    def __init__(\n        self,\n        images: np.ndarray,\n        targets: np.ndarray | None = None,\n        *,\n        indices: np.ndarray | None = None,\n    ) -> None:\n        self.images = images\n        self.targets = targets\n        self.indices = (\n            np.arange(len(images), dtype=np.int64)\n            if indices is None\n            else indices.astype(np.int64, copy=False)\n        )\n\n    def __len__(self) -> int:\n        return int(len(self.indices))\n\n    def __getitem__(self, index: int):\n        real_index = int(self.indices[index])\n        image = torch.from_numpy(self.images[real_index].astype(np.float32, copy=False)).unsqueeze(0)\n        if self.targets is None:\n            return image\n        target = torch.from_numpy(self.targets[real_index].astype(np.float32, copy=False))\n        return image, target\n\n\nclass MixedCuratedNoisyDataset(Dataset):\n    def __init__(\n        self,\n        *,\n        curated_images: np.ndarray,\n        curated_targets: np.ndarray,\n        curated_indices: np.ndarray,\n        noisy_images: np.ndarray,\n        noisy_targets: np.ndarray,\n        noisy_indices: np.ndarray,\n        curated_weight: float,\n        noisy_weight: float,\n        augment: bool,\n        gaussian_noise_std: float,\n    ) -> None:\n        self.curated_images = curated_images\n        self.curated_targets = curated_targets\n        self.curated_indices = curated_indices.astype(np.int64, copy=False)\n        self.noisy_images = noisy_images\n        self.noisy_targets = noisy_targets\n        self.noisy_indices = noisy_indices.astype(np.int64, copy=False)\n        self.curated_weight = float(curated_weight)\n        self.noisy_weight = float(noisy_weight)\n        self.augment = augment\n        self.gaussian_noise_std = float(gaussian_noise_std)\n\n    @property\n    def curated_len(self) -> int:\n        return int(len(self.curated_indices))\n\n    @property\n    def noisy_len(self) -> int:\n        return int(len(self.noisy_indices))\n\n    def __len__(self) -> int:\n        return self.curated_len + self.noisy_len\n\n    def __getitem__(self, index: int):\n        if index < self.curated_len:\n            real_index = int(self.curated_indices[index])\n            image = self.curated_images[real_index]\n            target = self.curated_targets[real_index]\n            weight = self.curated_weight\n        else:\n            noisy_index = int(self.noisy_indices[index - self.curated_len])\n            image = self.noisy_images[noisy_index]\n            target = self.noisy_targets[noisy_index]\n            weight = self.noisy_weight\n\n        image_tensor = torch.from_numpy(image.astype(np.float32, copy=False)).unsqueeze(0)\n        if self.augment:\n            image_tensor = augment_logmel_image(\n                image_tensor,\n                gaussian_noise_std=self.gaussian_noise_std,\n            )\n        target_tensor = torch.from_numpy(target.astype(np.float32, copy=False))\n        weight_tensor = torch.tensor(weight, dtype=torch.float32)\n        return image_tensor, target_tensor, weight_tensor\n\n\nclass HalfCuratedHalfNoisyBatchSampler(Sampler[list[int]]):\n    def __init__(\n        self,\n        *,\n        curated_len: int,\n        noisy_len: int,\n        batch_size: int,\n        seed: int,\n        steps_per_epoch: int | None = None,\n    ) -> None:\n        if curated_len <= 0:\n            raise ValueError(\"curated_len must be positive\")\n        if noisy_len <= 0:\n            raise ValueError(\"noisy_len must be positive\")\n        if batch_size < 2:\n            raise ValueError(\"batch_size must be at least 2\")\n        self.curated_len = int(curated_len)\n        self.noisy_len = int(noisy_len)\n        self.batch_size = int(batch_size)\n        self.curated_per_batch = self.batch_size // 2\n        self.noisy_per_batch = self.batch_size - self.curated_per_batch\n        self.steps_per_epoch = int(\n            steps_per_epoch\n            if steps_per_epoch is not None\n            else math.ceil(self.curated_len / self.curated_per_batch)\n        )\n        self.seed = int(seed)\n        self._epoch = 0\n\n    def __len__(self) -> int:\n        return self.steps_per_epoch\n\n    def __iter__(self) -> Iterable[list[int]]:\n        rng = np.random.default_rng(self.seed + self._epoch)\n        self._epoch += 1\n\n        curated_needed = self.steps_per_epoch * self.curated_per_batch\n        noisy_needed = self.steps_per_epoch * self.noisy_per_batch\n        curated_indices = _sample_epoch_indices(rng, self.curated_len, curated_needed)\n        noisy_indices = _sample_epoch_indices(rng, self.noisy_len, noisy_needed) + self.curated_len\n\n        for step in range(self.steps_per_epoch):\n            c_start = step * self.curated_per_batch\n            n_start = step * self.noisy_per_batch\n            batch = np.concatenate(\n                [\n                    curated_indices[c_start : c_start + self.curated_per_batch],\n                    noisy_indices[n_start : n_start + self.noisy_per_batch],\n                ]\n            )\n            rng.shuffle(batch)\n            yield batch.astype(np.int64).tolist()\n\n\ndef _sample_epoch_indices(\n    rng: np.random.Generator,\n    population_size: int,\n    needed: int,\n) -> np.ndarray:\n    if needed <= population_size:\n        return rng.permutation(population_size)[:needed]\n    repeats = needed // population_size\n    remainder = needed % population_size\n    chunks = [rng.permutation(population_size) for _ in range(repeats)]\n    if remainder:\n        chunks.append(rng.permutation(population_size)[:remainder])\n    return np.concatenate(chunks)\n\n\ndef augment_logmel_image(\n    image: torch.Tensor,\n    *,\n    gaussian_noise_std: float,\n) -> torch.Tensor:\n    if torch.rand(()) < 0.5:\n        shift = int(torch.randint(-32, 33, ()).item())\n        image = torch.roll(image, shifts=shift, dims=2)\n    if torch.rand(()) < 0.5:\n        width = int(torch.randint(8, 48, ()).item())\n        start = int(torch.randint(0, max(1, image.shape[2] - width + 1), ()).item())\n        image[:, :, start : start + width] = 0.0\n    if torch.rand(()) < 0.5:\n        width = int(torch.randint(4, 16, ()).item())\n        start = int(torch.randint(0, max(1, image.shape[1] - width + 1), ()).item())\n        image[:, start : start + width, :] = 0.0\n    if gaussian_noise_std > 0.0 and torch.rand(()) < 0.5:\n        image = image + torch.randn_like(image) * gaussian_noise_std\n    return image\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Fine-tune E100 FAT2019 branches with curated + noisy batches.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--work-dir\", type=Path, default=Path(\"proyecto_actual_v2/codigo/work\"))\n    parser.add_argument(\"--branches\", default=\"all\", help=\"Comma-separated branch names or 'all'.\")\n    parser.add_argument(\"--epochs\", type=int, default=FINE_TUNE.epochs)\n    parser.add_argument(\"--lr\", type=float, default=FINE_TUNE.lr)\n    parser.add_argument(\"--min-lr\", type=float, default=FINE_TUNE.min_lr)\n    parser.add_argument(\"--noisy-loss-weight\", type=float, default=FINE_TUNE.noisy_loss_weight)\n    parser.add_argument(\"--curated-loss-weight\", type=float, default=FINE_TUNE.curated_loss_weight)\n    parser.add_argument(\"--gaussian-noise-std\", type=float, default=FINE_TUNE.gaussian_noise_std)\n    parser.add_argument(\"--seed\", type=int, default=FINE_TUNE.seed)\n    parser.add_argument(\"--num-workers\", type=int, default=FINE_TUNE.num_workers)\n    parser.add_argument(\"--max-curated-train\", type=int, default=None)\n    parser.add_argument(\"--max-noisy\", type=int, default=None)\n    parser.add_argument(\"--submission-checkpoint\", choices=(\"final\", \"best\"), default=\"final\")\n    return parser\n\n\ndef _torch_load(path: Path, *, map_location: torch.device):\n    try:\n        return torch.load(path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(path, map_location=map_location)\n\n\ndef _load_cache(\n    branch: BranchConfig,\n    *,\n    split: str,\n    data_dir: Path,\n) -> tuple[np.ndarray, np.ndarray]:\n    if branch.frames == 1024 and branch.cache_tag is None:\n        memmap_stem = data_dir / f\"{split}_logmel_image_m{branch.n_mels}_f{branch.frames}_x\"\n        memmap_path = memmap_stem.with_suffix(\".npy\")\n        fnames_path = data_dir / f\"{memmap_stem.name}_fnames.txt\"\n        if memmap_path.exists() and fnames_path.exists():\n            x = np.load(memmap_path, mmap_mode=\"r\")\n            fnames = np.array(\n                [line.strip() for line in fnames_path.read_text().splitlines() if line.strip()],\n                dtype=str,\n            )\n            return x, fnames\n\n    path = logmel_image_cache_path(\n        data_dir,\n        split=split,\n        n_mels=branch.n_mels,\n        frames=branch.frames,\n        tag=branch.cache_tag,\n    )\n    if not path.exists():\n        raise FileNotFoundError(f\"missing cache: {path}\")\n    cache = np.load(path, allow_pickle=False)\n    return cache[\"x\"], cache[\"fnames\"].astype(str)\n\n\ndef _assert_order(expected: list[str], actual: np.ndarray, *, label: str) -> None:\n    actual_list = actual.astype(str).tolist()\n    if expected != actual_list:\n        mismatch = next(\n            (\n                index\n                for index, (left, right) in enumerate(zip(expected, actual_list))\n                if left != right\n            ),\n            None,\n        )\n        raise ValueError(f\"{label} cache order mismatch at row {mismatch}: expected labels order\")\n\n\ndef _predict_scores(\n    model: nn.Module,\n    loader: DataLoader,\n    *,\n    device: torch.device,\n) -> np.ndarray:\n    model.eval()\n    logits: list[np.ndarray] = []\n    with torch.no_grad():\n        for batch in loader:\n            images = batch[0] if isinstance(batch, (list, tuple)) else batch\n            output = model(images.to(device, non_blocking=True))\n            logits.append(output.detach().cpu().numpy())\n    return sigmoid_numpy(np.vstack(logits))\n\n\ndef _weighted_train_epoch(\n    model: nn.Module,\n    loader: DataLoader,\n    *,\n    criterion: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n) -> float:\n    model.train()\n    weighted_loss_sum = 0.0\n    weight_sum = 0.0\n    for images, targets, sample_weights in loader:\n        images = images.to(device, non_blocking=True)\n        targets = targets.to(device, non_blocking=True)\n        sample_weights = sample_weights.to(device, non_blocking=True)\n\n        optimizer.zero_grad(set_to_none=True)\n        logits = model(images)\n        loss_by_class = criterion(logits, targets)\n        loss_by_sample = loss_by_class.mean(dim=1)\n        weighted_losses = loss_by_sample * sample_weights\n        loss = weighted_losses.sum() / sample_weights.sum().clamp_min(1e-6)\n        loss.backward()\n        optimizer.step()\n\n        weighted_loss_sum += float(weighted_losses.detach().sum().cpu())\n        weight_sum += float(sample_weights.detach().sum().cpu())\n    return weighted_loss_sum / max(weight_sum, 1e-6)\n\n\ndef _model_from_checkpoint(checkpoint: dict[str, object], *, num_classes: int, device: torch.device) -> SmallLogmelCnn:\n    model = SmallLogmelCnn(\n        num_classes=num_classes,\n        architecture=str(checkpoint.get(\"architecture\", \"standard\")),\n        activation=str(checkpoint.get(\"activation\", \"silu\")),\n        block_dropout=float(checkpoint.get(\"block_dropout\", 0.0)),\n        head_hidden=int(checkpoint.get(\"head_hidden\", 0)),\n        head_dropout=float(checkpoint.get(\"head_dropout\", 0.35)),\n    ).to(device)\n    model.load_state_dict(checkpoint[\"model_state\"])\n    return model\n\n\ndef _save_checkpoint(\n    path: Path,\n    *,\n    model: nn.Module,\n    source_checkpoint: dict[str, object],\n    epoch: int,\n    valid_lwlrap: float,\n    branch: BranchConfig,\n    args: argparse.Namespace,\n) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = dict(source_checkpoint)\n    payload.update(\n        {\n            \"model_state\": model.state_dict(),\n            \"epoch\": int(epoch),\n            \"valid_lwlrap\": float(valid_lwlrap),\n            \"fine_tune_source_epoch\": int(source_checkpoint.get(\"epoch\", 0)),\n            \"fine_tune_branch\": branch.name,\n            \"fine_tune_epochs\": int(args.epochs),\n            \"fine_tune_lr\": float(args.lr),\n            \"fine_tune_scheduler\": \"cosine\",\n            \"fine_tune_curated_loss_weight\": float(args.curated_loss_weight),\n            \"fine_tune_noisy_loss_weight\": float(args.noisy_loss_weight),\n            \"fine_tune_gaussian_noise_std\": float(args.gaussian_noise_std),\n        }\n    )\n    torch.save(payload, path)\n\n\ndef _write_branch_metadata(\n    path: Path,\n    *,\n    branch: BranchConfig,\n    source_metadata: dict[str, object],\n    result: BranchRunResult,\n    args: argparse.Namespace,\n    rows: dict[str, int],\n    steps_per_epoch: int,\n) -> None:\n    lines = [\n        f\"# {branch.name} noisy fine-tune\",\n        \"\",\n        \"| Metric | Value |\",\n        \"|---|---:|\",\n        f\"| Baseline validation lwlrap | {result.baseline_lwlrap:.6f} |\",\n        f\"| Best validation lwlrap | {result.best_lwlrap:.6f} |\",\n        f\"| Final validation lwlrap | {result.final_lwlrap:.6f} |\",\n        f\"| Delta best-baseline | {result.best_lwlrap - result.baseline_lwlrap:+.6f} |\",\n        f\"| Best epoch | {result.best_epoch} |\",\n        f\"| Epochs requested | {args.epochs} |\",\n        f\"| LR | {args.lr} |\",\n        f\"| Noisy loss weight | {args.noisy_loss_weight} |\",\n        f\"| Steps per epoch | {steps_per_epoch} |\",\n        \"\",\n        \"## Source config\",\n        \"\",\n        \"```json\",\n        json.dumps(source_metadata, indent=2, sort_keys=True),\n        \"```\",\n        \"\",\n        \"## Rows\",\n        \"\",\n        \"```json\",\n        json.dumps(rows, indent=2, sort_keys=True),\n        \"```\",\n        \"\",\n    ]\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(\"\\n\".join(lines))\n\n\ndef run_branch(branch: BranchConfig, args: argparse.Namespace) -> BranchRunResult:\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    torch.backends.cudnn.benchmark = True\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    data_dir = args.data_dir\n    work_dir = args.work_dir\n    sample_submission = read_sample_submission(data_dir / \"sample_submission.csv\")\n    label_columns = label_columns_from_sample(sample_submission)\n\n    curated_labels = load_training_labels(data_dir / \"train_curated.csv\")\n    noisy_labels = load_training_labels(data_dir / \"train_noisy.csv\")\n    curated_y = dataframe_to_multihot(curated_labels, label_columns).astype(np.float32)\n    noisy_y = dataframe_to_multihot(noisy_labels, label_columns).astype(np.float32)\n\n    curated_x, curated_fnames = _load_cache(branch, split=\"curated\", data_dir=data_dir)\n    noisy_x, noisy_fnames = _load_cache(branch, split=\"noisy\", data_dir=data_dir)\n    _assert_order(curated_labels[\"fname\"].astype(str).tolist(), curated_fnames, label=\"curated\")\n    _assert_order(noisy_labels[\"fname\"].astype(str).tolist(), noisy_fnames, label=\"noisy\")\n\n    train_indices, valid_indices = make_train_valid_indices(\n        num_rows=len(curated_x),\n        test_size=FINE_TUNE.validation_test_size,\n        seed=args.seed,\n        full_train=False,\n    )\n    if args.max_curated_train is not None:\n        train_indices = train_indices[: args.max_curated_train]\n    noisy_indices = np.arange(len(noisy_x), dtype=np.int64)\n    if args.max_noisy is not None:\n        noisy_indices = noisy_indices[: args.max_noisy]\n        noisy_labels = noisy_labels.head(args.max_noisy).copy()\n\n    source_checkpoint_path = branch.source_checkpoint(work_dir)\n    source_metadata_path = branch.source_metadata(work_dir)\n    if not source_checkpoint_path.exists():\n        raise FileNotFoundError(\n            f\"missing source checkpoint for {branch.name}: {source_checkpoint_path}. \"\n            \"Run the curated E100 stage first.\"\n        )\n    if not source_metadata_path.exists():\n        raise FileNotFoundError(\n            f\"missing source metadata for {branch.name}: {source_metadata_path}. \"\n            \"Run the curated E100 stage first.\"\n        )\n    source_checkpoint = _torch_load(source_checkpoint_path, map_location=device)\n    source_metadata = json.loads(source_metadata_path.read_text())\n    if int(source_checkpoint.get(\"frames\", branch.frames)) != branch.frames:\n        raise ValueError(f\"{branch.name}: checkpoint frames do not match branch config\")\n    if str(source_checkpoint.get(\"cache_tag\", branch.cache_tag)) != str(branch.cache_tag):\n        raise ValueError(f\"{branch.name}: checkpoint cache tag does not match branch config\")\n\n    model = _model_from_checkpoint(\n        source_checkpoint,\n        num_classes=len(label_columns),\n        device=device,\n    )\n\n    valid_loader = DataLoader(\n        IndexedLogmelDataset(curated_x, curated_y, indices=valid_indices),\n        batch_size=branch.batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=device.type == \"cuda\",\n    )\n    baseline_scores = _predict_scores(model, valid_loader, device=device)\n    baseline_lwlrap = calculate_overall_lwlrap(curated_y[valid_indices], baseline_scores)\n\n    dataset = MixedCuratedNoisyDataset(\n        curated_images=curated_x,\n        curated_targets=curated_y,\n        curated_indices=train_indices,\n        noisy_images=noisy_x,\n        noisy_targets=noisy_y,\n        noisy_indices=noisy_indices,\n        curated_weight=args.curated_loss_weight,\n        noisy_weight=args.noisy_loss_weight,\n        augment=True,\n        gaussian_noise_std=args.gaussian_noise_std,\n    )\n    steps_per_epoch = math.ceil(len(train_indices) / (branch.batch_size // 2))\n    train_loader = DataLoader(\n        dataset,\n        batch_sampler=HalfCuratedHalfNoisyBatchSampler(\n            curated_len=dataset.curated_len,\n            noisy_len=dataset.noisy_len,\n            batch_size=branch.batch_size,\n            seed=args.seed,\n            steps_per_epoch=steps_per_epoch,\n        ),\n        num_workers=args.num_workers,\n        pin_memory=device.type == \"cuda\",\n    )\n\n    pos_weight = torch.from_numpy(compute_pos_weight(curated_y[train_indices])).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=\"none\")\n    optimizer = build_optimizer(\n        model.parameters(),\n        optimizer_name=str(source_metadata.get(\"optimizer\", \"adamw\")),\n        learning_rate=args.lr,\n        weight_decay=float(source_metadata.get(\"weight_decay\", 1e-4)),\n    )\n    scheduler = build_scheduler(\n        optimizer,\n        scheduler_name=\"cosine\",\n        epochs=args.epochs,\n        plateau_patience=2,\n        plateau_factor=0.5,\n        lr_milestones=[],\n        min_lr=args.min_lr,\n    )\n\n    branch_run_dir = branch.run_dir(work_dir)\n    model_dir = branch_run_dir / \"models\"\n    experiments_dir = branch_run_dir / \"experiments\"\n    submissions_dir = branch_run_dir / \"submissions\"\n    best_model_path = model_dir / \"small_logmel_cnn_best.pt\"\n    final_model_path = model_dir / \"small_logmel_cnn_final.pt\"\n    history: list[dict[str, float | int]] = []\n    best_lwlrap = baseline_lwlrap\n    best_epoch = 0\n    _save_checkpoint(\n        best_model_path,\n        model=model,\n        source_checkpoint=source_checkpoint,\n        epoch=0,\n        valid_lwlrap=baseline_lwlrap,\n        branch=branch,\n        args=args,\n    )\n\n    print(\n        f\"{branch.name}: baseline_lwlrap={baseline_lwlrap:.6f} \"\n        f\"device={device} steps_per_epoch={steps_per_epoch}\",\n        flush=True,\n    )\n    final_lwlrap = baseline_lwlrap\n    for epoch in range(1, args.epochs + 1):\n        learning_rate = float(optimizer.param_groups[0][\"lr\"])\n        train_loss = _weighted_train_epoch(\n            model,\n            train_loader,\n            criterion=criterion,\n            optimizer=optimizer,\n            device=device,\n        )\n        valid_scores = _predict_scores(model, valid_loader, device=device)\n        valid_lwlrap = calculate_overall_lwlrap(curated_y[valid_indices], valid_scores)\n        final_lwlrap = valid_lwlrap\n        scheduler.step()\n        history.append(\n            {\n                \"epoch\": epoch,\n                \"train_loss\": float(train_loss),\n                \"valid_lwlrap\": float(valid_lwlrap),\n                \"learning_rate\": learning_rate,\n            }\n        )\n        print(\n            f\"{branch.name} epoch {epoch}: \"\n            f\"loss={train_loss:.5f} valid_lwlrap={valid_lwlrap:.6f} lr={learning_rate:.8f}\",\n            flush=True,\n        )\n        if valid_lwlrap > best_lwlrap:\n            best_lwlrap = valid_lwlrap\n            best_epoch = epoch\n            _save_checkpoint(\n                best_model_path,\n                model=model,\n                source_checkpoint=source_checkpoint,\n                epoch=epoch,\n                valid_lwlrap=valid_lwlrap,\n                branch=branch,\n                args=args,\n            )\n\n    _save_checkpoint(\n        final_model_path,\n        model=model,\n        source_checkpoint=source_checkpoint,\n        epoch=args.epochs,\n        valid_lwlrap=final_lwlrap,\n        branch=branch,\n        args=args,\n    )\n\n    best_checkpoint = _torch_load(best_model_path, map_location=device)\n    best_model = _model_from_checkpoint(best_checkpoint, num_classes=len(label_columns), device=device)\n    valid_scores_best = _predict_scores(best_model, valid_loader, device=device)\n    final_checkpoint = _torch_load(final_model_path, map_location=device)\n    final_model = _model_from_checkpoint(final_checkpoint, num_classes=len(label_columns), device=device)\n    valid_scores_final = _predict_scores(final_model, valid_loader, device=device)\n\n    del train_loader, dataset, noisy_x, noisy_y, optimizer, criterion, best_model\n    gc.collect()\n    if device.type == \"cuda\":\n        torch.cuda.empty_cache()\n\n    test_x, test_fnames = _load_cache(branch, split=\"test\", data_dir=data_dir)\n    _assert_order(sample_submission[\"fname\"].astype(str).tolist(), test_fnames, label=\"test\")\n    prediction_model = final_model if args.submission_checkpoint == \"final\" else _model_from_checkpoint(\n        best_checkpoint,\n        num_classes=len(label_columns),\n        device=device,\n    )\n    test_loader = DataLoader(\n        LogmelDataset(test_x, augment=False),\n        batch_size=branch.batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=device.type == \"cuda\",\n    )\n    test_scores = _predict_scores(prediction_model, test_loader, device=device)\n    submission = pd.DataFrame(np.clip(test_scores, 0.0, 1.0), columns=label_columns)\n    submission.insert(0, \"fname\", sample_submission[\"fname\"].astype(str).to_numpy())\n    submission_path = submissions_dir / \"small_logmel_cnn.csv\"\n    write_submission(submission, submission_path, label_columns)\n\n    experiments_dir.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(history).to_csv(experiments_dir / \"history.csv\", index=False)\n    np.save(experiments_dir / \"valid_scores_best.npy\", valid_scores_best.astype(np.float32))\n    np.save(experiments_dir / \"valid_scores_final.npy\", valid_scores_final.astype(np.float32))\n    np.save(experiments_dir / \"valid_targets.npy\", curated_y[valid_indices].astype(np.float32))\n    pd.DataFrame({\"fname\": curated_labels.iloc[valid_indices][\"fname\"].astype(str)}).to_csv(\n        experiments_dir / \"valid_fnames.csv\",\n        index=False,\n    )\n\n    result = BranchRunResult(\n        branch=branch.name,\n        baseline_lwlrap=float(baseline_lwlrap),\n        best_lwlrap=float(best_lwlrap),\n        final_lwlrap=float(final_lwlrap),\n        best_epoch=int(best_epoch),\n        submission_path=submission_path,\n        best_model_path=best_model_path,\n        final_model_path=final_model_path,\n    )\n    rows = {\n        \"curated_total\": int(len(curated_x)),\n        \"curated_train\": int(len(train_indices)),\n        \"curated_valid\": int(len(valid_indices)),\n        \"noisy_total\": int(len(noisy_indices)),\n        \"test\": int(len(test_x)),\n    }\n    _write_branch_metadata(\n        experiments_dir / \"metadata.md\",\n        branch=branch,\n        source_metadata=source_metadata,\n        result=result,\n        args=args,\n        rows=rows,\n        steps_per_epoch=steps_per_epoch,\n    )\n    (experiments_dir / \"metadata.json\").write_text(\n        json.dumps(\n            {\n                \"branch\": branch.name,\n                \"baseline_lwlrap\": result.baseline_lwlrap,\n                \"best_lwlrap\": result.best_lwlrap,\n                \"final_lwlrap\": result.final_lwlrap,\n                \"best_epoch\": result.best_epoch,\n                \"submission_path\": str(submission_path),\n                \"best_model_path\": str(best_model_path),\n                \"final_model_path\": str(final_model_path),\n                \"rows\": rows,\n                \"steps_per_epoch\": steps_per_epoch,\n                \"epochs\": args.epochs,\n                \"lr\": args.lr,\n                \"scheduler\": \"cosine\",\n                \"noisy_loss_weight\": args.noisy_loss_weight,\n                \"curated_loss_weight\": args.curated_loss_weight,\n                \"gaussian_noise_std\": args.gaussian_noise_std,\n                \"submission_checkpoint\": args.submission_checkpoint,\n                \"source_checkpoint\": str(source_checkpoint_path),\n                \"source_metadata\": str(source_metadata_path),\n            },\n            indent=2,\n        )\n        + \"\\n\"\n    )\n    print(\n        f\"{branch.name}: best_lwlrap={result.best_lwlrap:.6f} \"\n        f\"final_lwlrap={result.final_lwlrap:.6f} wrote={submission_path}\",\n        flush=True,\n    )\n    return result\n\n\ndef _select_branches(raw: str) -> list[BranchConfig]:\n    if raw == \"all\":\n        return list(BRANCHES)\n    by_name = {branch.name: branch for branch in BRANCHES}\n    selected: list[BranchConfig] = []\n    for name in [part.strip() for part in raw.split(\",\") if part.strip()]:\n        if name not in by_name:\n            raise ValueError(f\"unknown branch {name!r}; choices are {sorted(by_name)}\")\n        selected.append(by_name[name])\n    if not selected:\n        raise ValueError(\"expected at least one branch\")\n    return selected\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    if args.epochs <= 0:\n        raise ValueError(\"--epochs must be positive\")\n    if args.lr <= 0.0:\n        raise ValueError(\"--lr must be positive\")\n    if args.min_lr <= 0.0:\n        raise ValueError(\"--min-lr must be positive\")\n    if not 0.0 < args.noisy_loss_weight <= args.curated_loss_weight:\n        raise ValueError(\"--noisy-loss-weight must be in (0, curated_loss_weight]\")\n    if args.gaussian_noise_std < 0.0:\n        raise ValueError(\"--gaussian-noise-std must be non-negative\")\n\n    results = []\n    for branch in _select_branches(args.branches):\n        results.append(run_branch(branch, args))\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    print(\"finetune_mixed_noisy_ok\")\n    for result in results:\n        print(\n            f\"{result.branch}: baseline={result.baseline_lwlrap:.6f} \"\n            f\"best={result.best_lwlrap:.6f} final={result.final_lwlrap:.6f}\"\n        )\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "train_logmel_cnn.py": "from __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom collections.abc import Iterable\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\n\nif __package__ is None or __package__ == \"\":\n    sys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom build_logmel_image_cache import logmel_image_cache_path\nfrom fat2019.data import load_training_labels\nfrom fat2019.labels import dataframe_to_multihot\nfrom fat2019.metrics import calculate_overall_lwlrap\nfrom fat2019.neural_helpers import (\n    compute_pos_weight,\n    make_train_valid_indices,\n    sigmoid_numpy,\n)\nfrom fat2019.submission import (\n    label_columns_from_sample,\n    read_sample_submission,\n    write_submission,\n)\n\n\n@dataclass(frozen=True)\nclass EpochResult:\n    epoch: int\n    train_loss: float\n    valid_lwlrap: float | None\n    learning_rate: float\n\n\nclass LogmelDataset(Dataset):\n    def __init__(\n        self,\n        images: np.ndarray,\n        targets: np.ndarray | None = None,\n        *,\n        augment: bool = False,\n        time_reverse_probability: float = 0.0,\n        contrast_strength: float = 0.0,\n    ) -> None:\n        self._images = images\n        self._targets = targets\n        self._augment = augment\n        self._time_reverse_probability = time_reverse_probability\n        self._contrast_strength = contrast_strength\n\n    def __len__(self) -> int:\n        return int(len(self._images))\n\n    def __getitem__(self, index: int):\n        image = torch.from_numpy(self._images[index].astype(np.float32, copy=False)).unsqueeze(0)\n        if self._augment:\n            image = _augment_image(\n                image,\n                time_reverse_probability=self._time_reverse_probability,\n                contrast_strength=self._contrast_strength,\n            )\n        if self._targets is None:\n            return image\n        target = torch.from_numpy(self._targets[index].astype(np.float32, copy=False))\n        return image, target\n\n\ndef _scale_contrast(image: torch.Tensor, *, factor: float) -> torch.Tensor:\n    mean = image.mean()\n    return (image - mean) * factor + mean\n\n\ndef _augment_image(\n    image: torch.Tensor,\n    *,\n    apply_spec_augment: bool = True,\n    time_reverse_probability: float = 0.0,\n    contrast_strength: float = 0.0,\n) -> torch.Tensor:\n    if time_reverse_probability > 0.0 and torch.rand(()) < time_reverse_probability:\n        image = image.flip(dims=(2,))\n    if contrast_strength > 0.0:\n        factor = float(torch.empty(()).uniform_(1.0 - contrast_strength, 1.0 + contrast_strength))\n        image = _scale_contrast(image, factor=factor)\n    if apply_spec_augment:\n        if torch.rand(()) < 0.5:\n            shift = int(torch.randint(-32, 33, ()).item())\n            image = torch.roll(image, shifts=shift, dims=2)\n        if torch.rand(()) < 0.5:\n            width = int(torch.randint(8, 48, ()).item())\n            start = int(torch.randint(0, max(1, image.shape[2] - width + 1), ()).item())\n            image[:, :, start : start + width] = 0.0\n        if torch.rand(()) < 0.5:\n            width = int(torch.randint(4, 16, ()).item())\n            start = int(torch.randint(0, max(1, image.shape[1] - width + 1), ()).item())\n            image[:, start : start + width, :] = 0.0\n    return image\n\n\nclass HiddenLinear(nn.Linear):\n    pass\n\n\nclass TemporalBigruHead(nn.Module):\n    def __init__(\n        self,\n        *,\n        input_channels: int,\n        hidden_size: int,\n        num_classes: int,\n        dropout: float,\n    ) -> None:\n        super().__init__()\n        self.gru = nn.GRU(\n            input_channels,\n            hidden_size,\n            batch_first=True,\n            bidirectional=True,\n        )\n        self.dropout = nn.Dropout(p=dropout)\n        self.classifier = nn.Linear(hidden_size * 4, num_classes)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.ndim != 4:\n            raise ValueError(f\"expected CNN features with shape (batch, channels, freq, time), got {x.shape}\")\n        sequence = x.mean(dim=2).transpose(1, 2)\n        outputs, _hidden = self.gru(sequence)\n        pooled = torch.cat([outputs.mean(dim=1), outputs.amax(dim=1)], dim=1)\n        return self.classifier(self.dropout(pooled))\n\n\nclass DepthwiseSeparableConv(nn.Module):\n    def __init__(\n        self,\n        in_channels: int,\n        out_channels: int,\n        *,\n        activation: str,\n    ) -> None:\n        super().__init__()\n        self.layers = nn.Sequential(\n            nn.Conv2d(\n                in_channels,\n                in_channels,\n                kernel_size=3,\n                padding=1,\n                groups=in_channels,\n                bias=False,\n            ),\n            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            _activation_layer(activation),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.layers(x)\n\n\nclass SeparableResidualBlock(nn.Module):\n    def __init__(\n        self,\n        in_channels: int,\n        out_channels: int,\n        *,\n        activation: str,\n    ) -> None:\n        super().__init__()\n        self.main = nn.Sequential(\n            DepthwiseSeparableConv(in_channels, out_channels, activation=activation),\n            DepthwiseSeparableConv(out_channels, out_channels, activation=activation),\n            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),\n        )\n        self.residual = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=2, bias=False),\n            nn.BatchNorm2d(out_channels),\n        )\n        self.activation = _activation_layer(activation)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.activation(self.main(x) + self.residual(x))\n\n\nclass SqueezeExcitation(nn.Module):\n    def __init__(self, channels: int, *, reduction: int = 16) -> None:\n        super().__init__()\n        reduced_channels = max(1, channels // reduction)\n        self.layers = nn.Sequential(\n            nn.AdaptiveAvgPool2d((1, 1)),\n            nn.Flatten(),\n            nn.Linear(channels, reduced_channels),\n            nn.ReLU(inplace=True),\n            nn.Linear(reduced_channels, channels),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        scale = self.layers(x).view(x.shape[0], x.shape[1], 1, 1)\n        return x * scale\n\n\nclass SmallLogmelCnn(nn.Module):\n    def __init__(\n        self,\n        num_classes: int,\n        *,\n        architecture: str = \"standard\",\n        activation: str = \"silu\",\n        block_dropout: float = 0.0,\n        head_hidden: int = 0,\n        head_dropout: float = 0.35,\n    ) -> None:\n        super().__init__()\n        temporal_head = False\n        if architecture == \"standard\":\n            feature_channels = 192\n            self.features = nn.Sequential(\n                _conv_block(1, 32, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(2),\n                _conv_block(32, 64, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(2),\n                _conv_block(64, 128, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(2),\n                _conv_block(128, 192, activation=activation, dropout=block_dropout),\n                nn.AdaptiveAvgPool2d((1, 1)),\n            )\n        elif architecture == \"separable_residual\":\n            feature_channels = 512\n            self.features = nn.Sequential(\n                nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1, bias=False),\n                nn.BatchNorm2d(64),\n                _activation_layer(activation),\n                SeparableResidualBlock(64, 128, activation=activation),\n                SeparableResidualBlock(128, 256, activation=activation),\n                SeparableResidualBlock(256, 384, activation=activation),\n                DepthwiseSeparableConv(384, feature_channels, activation=activation),\n                nn.AdaptiveAvgPool2d((1, 1)),\n            )\n        elif architecture == \"separable_residual_se\":\n            feature_channels = 512\n            self.features = nn.Sequential(\n                nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1, bias=False),\n                nn.BatchNorm2d(64),\n                _activation_layer(activation),\n                SeparableResidualBlock(64, 128, activation=activation),\n                SqueezeExcitation(128),\n                SeparableResidualBlock(128, 256, activation=activation),\n                SqueezeExcitation(256),\n                SeparableResidualBlock(256, 384, activation=activation),\n                SqueezeExcitation(384),\n                DepthwiseSeparableConv(384, feature_channels, activation=activation),\n                SqueezeExcitation(feature_channels),\n                nn.AdaptiveAvgPool2d((1, 1)),\n            )\n        elif architecture == \"separable_temporal_bigru\":\n            feature_channels = 512\n            temporal_head = True\n            self.features = nn.Sequential(\n                nn.Conv2d(1, 64, kernel_size=3, stride=2, padding=1, bias=False),\n                nn.BatchNorm2d(64),\n                _activation_layer(activation),\n                SeparableResidualBlock(64, 128, activation=activation),\n                SeparableResidualBlock(128, 256, activation=activation),\n                SeparableResidualBlock(256, 384, activation=activation),\n                DepthwiseSeparableConv(384, feature_channels, activation=activation),\n            )\n        elif architecture == \"temporal_bigru\":\n            feature_channels = 128\n            temporal_head = True\n            self.features = nn.Sequential(\n                _conv_block(1, 32, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(kernel_size=(2, 2)),\n                _conv_block(32, 64, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(kernel_size=(2, 2)),\n                _conv_block(64, feature_channels, activation=activation, dropout=block_dropout),\n                nn.MaxPool2d(kernel_size=(2, 1)),\n            )\n        else:\n            raise ValueError(f\"unknown architecture: {architecture}\")\n\n        if temporal_head:\n            self.classifier = TemporalBigruHead(\n                input_channels=feature_channels,\n                hidden_size=64,\n                num_classes=num_classes,\n                dropout=head_dropout,\n            )\n        elif head_hidden > 0:\n            self.classifier = nn.Sequential(\n                nn.Flatten(),\n                HiddenLinear(feature_channels, head_hidden, bias=False),\n                nn.BatchNorm1d(head_hidden),\n                _activation_layer(activation),\n                nn.Dropout(p=head_dropout),\n                nn.Linear(head_hidden, num_classes),\n            )\n        else:\n            self.classifier = nn.Sequential(\n                nn.Flatten(),\n                nn.Dropout(p=head_dropout),\n                nn.Linear(feature_channels, num_classes),\n            )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.classifier(self.features(x))\n\n\ndef apply_he_initialization(module: nn.Module) -> None:\n    if isinstance(module, nn.Conv2d):\n        nn.init.kaiming_normal_(module.weight, mode=\"fan_in\", nonlinearity=\"relu\")\n        if module.bias is not None:\n            nn.init.zeros_(module.bias)\n    elif isinstance(module, HiddenLinear):\n        nn.init.kaiming_normal_(module.weight, mode=\"fan_in\", nonlinearity=\"relu\")\n        if module.bias is not None:\n            nn.init.zeros_(module.bias)\n    elif isinstance(module, nn.Linear):\n        nn.init.xavier_normal_(module.weight)\n        if module.bias is not None:\n            nn.init.zeros_(module.bias)\n    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):\n        if module.weight is not None:\n            nn.init.ones_(module.weight)\n        if module.bias is not None:\n            nn.init.zeros_(module.bias)\n\n\ndef _activation_layer(name: str) -> nn.Module:\n    if name == \"silu\":\n        return nn.SiLU(inplace=True)\n    if name == \"relu\":\n        return nn.ReLU(inplace=True)\n    raise ValueError(f\"unknown activation: {name}\")\n\n\ndef _conv_block(\n    in_channels: int,\n    out_channels: int,\n    *,\n    activation: str,\n    dropout: float,\n) -> nn.Sequential:\n    layers: list[nn.Module] = [\n        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),\n        nn.BatchNorm2d(out_channels),\n        _activation_layer(activation),\n        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),\n        nn.BatchNorm2d(out_channels),\n        _activation_layer(activation),\n    ]\n    if dropout > 0.0:\n        layers.append(nn.Dropout2d(p=dropout))\n    return nn.Sequential(*layers)\n\n\ndef build_scheduler(\n    optimizer: torch.optim.Optimizer,\n    *,\n    scheduler_name: str,\n    epochs: int,\n    plateau_patience: int,\n    plateau_factor: float,\n    lr_milestones: list[int],\n    min_lr: float = 1e-6,\n) -> torch.optim.lr_scheduler.LRScheduler | torch.optim.lr_scheduler.ReduceLROnPlateau:\n    if scheduler_name == \"cosine\":\n        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    if scheduler_name == \"plateau\":\n        return torch.optim.lr_scheduler.ReduceLROnPlateau(\n            optimizer,\n            mode=\"max\",\n            factor=plateau_factor,\n            patience=plateau_patience,\n            min_lr=min_lr,\n        )\n    if scheduler_name == \"multistep\":\n        if not lr_milestones:\n            raise ValueError(\"multistep scheduler requires at least one LR milestone\")\n        return torch.optim.lr_scheduler.MultiStepLR(\n            optimizer,\n            milestones=lr_milestones,\n            gamma=plateau_factor,\n        )\n    raise ValueError(f\"unknown scheduler: {scheduler_name}\")\n\n\ndef build_optimizer(\n    parameters: Iterable[nn.Parameter],\n    *,\n    optimizer_name: str,\n    learning_rate: float,\n    weight_decay: float,\n) -> torch.optim.Optimizer:\n    if optimizer_name == \"adamw\":\n        return torch.optim.AdamW(\n            parameters,\n            lr=learning_rate,\n            weight_decay=weight_decay,\n        )\n    if optimizer_name == \"adam\":\n        return torch.optim.Adam(parameters, lr=learning_rate)\n    raise ValueError(f\"unknown optimizer: {optimizer_name}\")\n\n\ndef parse_lr_milestones(raw_milestones: str) -> list[int]:\n    if not raw_milestones.strip():\n        return []\n    milestones = [int(value.strip()) for value in raw_milestones.split(\",\") if value.strip()]\n    if any(milestone <= 0 for milestone in milestones):\n        raise ValueError(\"LR milestones must be positive epoch numbers\")\n    if milestones != sorted(set(milestones)):\n        raise ValueError(\"LR milestones must be unique and sorted\")\n    return milestones\n\n\ndef _build_parser() -> argparse.ArgumentParser:\n    parser = argparse.ArgumentParser(description=\"Train a small CNN on fixed log-mel images.\")\n    parser.add_argument(\"--data-dir\", type=Path, default=Path(\"data\"))\n    parser.add_argument(\"--models-dir\", type=Path, default=Path(\"models/logmel_cnn\"))\n    parser.add_argument(\"--submissions-dir\", type=Path, default=Path(\"submissions/logmel_cnn\"))\n    parser.add_argument(\"--experiments-dir\", type=Path, default=Path(\"experiments/logmel_cnn\"))\n    parser.add_argument(\"--n-mels\", type=int, default=128)\n    parser.add_argument(\"--frames\", type=int, default=512)\n    parser.add_argument(\"--cache-tag\", default=None)\n    parser.add_argument(\"--epochs\", type=int, default=12)\n    parser.add_argument(\"--batch-size\", type=int, default=24)\n    parser.add_argument(\"--lr\", type=float, default=1e-3)\n    parser.add_argument(\"--weight-decay\", type=float, default=1e-4)\n    parser.add_argument(\"--optimizer\", choices=(\"adamw\", \"adam\"), default=\"adamw\")\n    parser.add_argument(\"--initializer\", choices=(\"default\", \"he_normal\"), default=\"default\")\n    parser.add_argument(\n        \"--architecture\",\n        choices=(\n            \"standard\",\n            \"separable_residual\",\n            \"separable_residual_se\",\n            \"temporal_bigru\",\n            \"separable_temporal_bigru\",\n        ),\n        default=\"standard\",\n    )\n    parser.add_argument(\"--activation\", choices=(\"silu\", \"relu\"), default=\"silu\")\n    parser.add_argument(\"--block-dropout\", type=float, default=0.0)\n    parser.add_argument(\"--head-hidden\", type=int, default=0)\n    parser.add_argument(\"--head-dropout\", type=float, default=0.35)\n    parser.add_argument(\"--time-reverse-probability\", type=float, default=0.0)\n    parser.add_argument(\"--contrast-strength\", type=float, default=0.0)\n    parser.add_argument(\n        \"--scheduler\",\n        choices=(\"cosine\", \"plateau\", \"multistep\"),\n        default=\"cosine\",\n    )\n    parser.add_argument(\"--lr-milestones\", default=\"\")\n    parser.add_argument(\"--plateau-patience\", type=int, default=2)\n    parser.add_argument(\"--plateau-factor\", type=float, default=0.5)\n    parser.add_argument(\"--min-lr\", type=float, default=1e-6)\n    parser.add_argument(\n        \"--early-stopping-patience\",\n        type=int,\n        default=0,\n        help=\"Stop after this many non-improving validation epochs; zero disables it.\",\n    )\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--test-size\", type=float, default=0.2)\n    parser.add_argument(\"--num-workers\", type=int, default=2)\n    parser.add_argument(\"--max-train\", type=int, default=None)\n    parser.add_argument(\n        \"--full-train\",\n        action=\"store_true\",\n        help=\"Train on all curated rows and skip holdout validation.\",\n    )\n    return parser\n\n\ndef _predict_logits(\n    model: nn.Module,\n    loader: DataLoader,\n    *,\n    device: torch.device,\n) -> np.ndarray:\n    model.eval()\n    logits: list[np.ndarray] = []\n    with torch.no_grad():\n        for batch in loader:\n            if isinstance(batch, (list, tuple)):\n                images = batch[0]\n            else:\n                images = batch\n            output = model(images.to(device, non_blocking=True))\n            logits.append(output.detach().cpu().numpy())\n    return np.vstack(logits)\n\n\ndef _train_epoch(\n    model: nn.Module,\n    loader: DataLoader,\n    *,\n    criterion: nn.Module,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n) -> float:\n    model.train()\n    total_loss = 0.0\n    total_rows = 0\n    for images, targets in loader:\n        images = images.to(device, non_blocking=True)\n        targets = targets.to(device, non_blocking=True)\n        optimizer.zero_grad(set_to_none=True)\n        logits = model(images)\n        loss = criterion(logits, targets)\n        loss.backward()\n        optimizer.step()\n        rows = int(images.shape[0])\n        total_loss += float(loss.detach().cpu()) * rows\n        total_rows += rows\n    return total_loss / max(1, total_rows)\n\n\ndef main() -> None:\n    args = _build_parser().parse_args()\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    torch.backends.cudnn.benchmark = True\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    print(f\"device={device}\")\n\n    sample_submission = read_sample_submission(args.data_dir / \"sample_submission.csv\")\n    label_columns = label_columns_from_sample(sample_submission)\n    labels = load_training_labels(args.data_dir / \"train_curated.csv\")\n    y = dataframe_to_multihot(labels, label_columns).astype(np.float32)\n\n    train_cache = np.load(\n        logmel_image_cache_path(\n            args.data_dir,\n            split=\"curated\",\n            n_mels=args.n_mels,\n            frames=args.frames,\n            tag=args.cache_tag,\n        ),\n        allow_pickle=False,\n    )\n    test_cache = np.load(\n        logmel_image_cache_path(\n            args.data_dir,\n            split=\"test\",\n            n_mels=args.n_mels,\n            frames=args.frames,\n            tag=args.cache_tag,\n        ),\n        allow_pickle=False,\n    )\n    x = train_cache[\"x\"]\n    x_test = test_cache[\"x\"]\n\n    if args.max_train is not None:\n        x = x[: args.max_train]\n        y = y[: args.max_train]\n        labels = labels.head(args.max_train).copy()\n\n    if len(x) != len(y):\n        raise ValueError(f\"feature/label row mismatch: {len(x)} vs {len(y)}\")\n\n    train_indices, valid_indices = make_train_valid_indices(\n        num_rows=len(x),\n        test_size=args.test_size,\n        seed=args.seed,\n        full_train=args.full_train,\n    )\n\n    train_loader = DataLoader(\n        LogmelDataset(\n            x[train_indices],\n            y[train_indices],\n            augment=True,\n            time_reverse_probability=args.time_reverse_probability,\n            contrast_strength=args.contrast_strength,\n        ),\n        batch_size=args.batch_size,\n        shuffle=True,\n        num_workers=args.num_workers,\n        pin_memory=device.type == \"cuda\",\n    )\n    valid_loader = None\n    if valid_indices.size > 0:\n        valid_loader = DataLoader(\n            LogmelDataset(x[valid_indices], y[valid_indices], augment=False),\n            batch_size=args.batch_size,\n            shuffle=False,\n            num_workers=args.num_workers,\n            pin_memory=device.type == \"cuda\",\n        )\n    test_loader = DataLoader(\n        LogmelDataset(x_test, augment=False),\n        batch_size=args.batch_size,\n        shuffle=False,\n        num_workers=args.num_workers,\n        pin_memory=device.type == \"cuda\",\n    )\n\n    if not 0.0 <= args.block_dropout < 1.0:\n        raise ValueError(\"block_dropout must be in [0, 1)\")\n    if not 0.0 <= args.head_dropout < 1.0:\n        raise ValueError(\"head_dropout must be in [0, 1)\")\n    if args.head_hidden < 0:\n        raise ValueError(\"head_hidden must be non-negative\")\n    if not 0.0 <= args.time_reverse_probability <= 1.0:\n        raise ValueError(\"time_reverse_probability must be in [0, 1]\")\n    if not 0.0 <= args.contrast_strength <= 1.0:\n        raise ValueError(\"contrast_strength must be in [0, 1]\")\n    lr_milestones = parse_lr_milestones(args.lr_milestones)\n\n    model = SmallLogmelCnn(\n        num_classes=len(label_columns),\n        architecture=args.architecture,\n        activation=args.activation,\n        block_dropout=args.block_dropout,\n        head_hidden=args.head_hidden,\n        head_dropout=args.head_dropout,\n    )\n    if args.initializer == \"he_normal\":\n        model.apply(apply_he_initialization)\n    model = model.to(device)\n    pos_weight = torch.from_numpy(compute_pos_weight(y[train_indices])).to(device)\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    optimizer = build_optimizer(\n        model.parameters(),\n        optimizer_name=args.optimizer,\n        learning_rate=args.lr,\n        weight_decay=args.weight_decay,\n    )\n    scheduler = build_scheduler(\n        optimizer,\n        scheduler_name=args.scheduler,\n        epochs=args.epochs,\n        plateau_patience=args.plateau_patience,\n        plateau_factor=args.plateau_factor,\n        lr_milestones=lr_milestones,\n        min_lr=args.min_lr,\n    )\n\n    args.models_dir.mkdir(parents=True, exist_ok=True)\n    args.submissions_dir.mkdir(parents=True, exist_ok=True)\n    args.experiments_dir.mkdir(parents=True, exist_ok=True)\n\n    best_lwlrap = -1.0\n    best_epoch = 0\n    best_model_path = args.models_dir / \"small_logmel_cnn_best.pt\"\n    history: list[EpochResult] = []\n    non_improving_epochs = 0\n    early_stopped = False\n    for epoch in range(1, args.epochs + 1):\n        learning_rate = float(optimizer.param_groups[0][\"lr\"])\n        train_loss = _train_epoch(\n            model,\n            train_loader,\n            criterion=criterion,\n            optimizer=optimizer,\n            device=device,\n        )\n        if valid_loader is None:\n            valid_lwlrap = None\n            history.append(\n                EpochResult(\n                    epoch=epoch,\n                    train_loss=train_loss,\n                    valid_lwlrap=None,\n                    learning_rate=learning_rate,\n                )\n            )\n            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):\n                scheduler.step(-train_loss)\n            else:\n                scheduler.step()\n            print(f\"epoch {epoch}: loss={train_loss:.5f} lr={learning_rate:.8f}\")\n            best_epoch = epoch\n            torch.save(\n                {\n                    \"model_state\": model.state_dict(),\n                    \"epoch\": epoch,\n                    \"valid_lwlrap\": None,\n                    \"label_columns\": label_columns,\n                    \"n_mels\": args.n_mels,\n                    \"frames\": args.frames,\n                    \"cache_tag\": args.cache_tag,\n                    \"full_train\": True,\n                    \"initializer\": args.initializer,\n                    \"architecture\": args.architecture,\n                    \"activation\": args.activation,\n                    \"block_dropout\": args.block_dropout,\n                    \"head_hidden\": args.head_hidden,\n                    \"head_dropout\": args.head_dropout,\n                    \"time_reverse_probability\": args.time_reverse_probability,\n                    \"contrast_strength\": args.contrast_strength,\n                },\n                best_model_path,\n            )\n            continue\n\n        valid_logits = _predict_logits(model, valid_loader, device=device)\n        valid_scores = sigmoid_numpy(valid_logits)\n        valid_lwlrap = calculate_overall_lwlrap(y[valid_indices], valid_scores)\n        history.append(\n            EpochResult(\n                epoch=epoch,\n                train_loss=train_loss,\n                valid_lwlrap=valid_lwlrap,\n                learning_rate=learning_rate,\n            )\n        )\n        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):\n            scheduler.step(valid_lwlrap)\n        else:\n            scheduler.step()\n        print(\n            f\"epoch {epoch}: loss={train_loss:.5f} \"\n            f\"valid_lwlrap={valid_lwlrap:.6f} lr={learning_rate:.8f}\"\n        )\n        if valid_lwlrap > best_lwlrap:\n            best_lwlrap = valid_lwlrap\n            best_epoch = epoch\n            non_improving_epochs = 0\n            torch.save(\n                {\n                    \"model_state\": model.state_dict(),\n                    \"epoch\": epoch,\n                    \"valid_lwlrap\": valid_lwlrap,\n                    \"label_columns\": label_columns,\n                    \"n_mels\": args.n_mels,\n                    \"frames\": args.frames,\n                    \"cache_tag\": args.cache_tag,\n                    \"full_train\": False,\n                    \"initializer\": args.initializer,\n                    \"architecture\": args.architecture,\n                    \"activation\": args.activation,\n                    \"block_dropout\": args.block_dropout,\n                    \"head_hidden\": args.head_hidden,\n                    \"head_dropout\": args.head_dropout,\n                    \"time_reverse_probability\": args.time_reverse_probability,\n                    \"contrast_strength\": args.contrast_strength,\n                },\n                best_model_path,\n            )\n        else:\n            non_improving_epochs += 1\n            if (\n                args.early_stopping_patience > 0\n                and non_improving_epochs >= args.early_stopping_patience\n            ):\n                early_stopped = True\n                print(f\"early_stopping epoch={epoch} best_epoch={best_epoch}\")\n                break\n\n    checkpoint = torch.load(best_model_path, map_location=device)\n    model.load_state_dict(checkpoint[\"model_state\"])\n    test_scores = sigmoid_numpy(_predict_logits(model, test_loader, device=device))\n    submission = pd.DataFrame(np.clip(test_scores, 0.0, 1.0), columns=label_columns)\n    submission.insert(0, \"fname\", sample_submission[\"fname\"].astype(str).to_numpy())\n    submission_path = args.submissions_dir / \"small_logmel_cnn.csv\"\n    write_submission(submission, submission_path, label_columns)\n\n    history_path = args.experiments_dir / \"small_logmel_cnn_history.csv\"\n    pd.DataFrame([result.__dict__ for result in history]).to_csv(history_path, index=False)\n    metadata_path = args.experiments_dir / \"small_logmel_cnn_metadata.json\"\n    metadata_path.write_text(\n        json.dumps(\n            {\n                \"best_lwlrap\": None if args.full_train else float(best_lwlrap),\n                \"best_epoch\": int(best_epoch),\n                \"device\": str(device),\n                \"rows\": int(len(x)),\n                \"train_rows\": int(len(train_indices)),\n                \"valid_rows\": int(len(valid_indices)),\n                \"labels\": len(label_columns),\n                \"n_mels\": args.n_mels,\n                \"frames\": args.frames,\n                \"cache_tag\": args.cache_tag,\n                \"seed\": args.seed,\n                \"test_size\": args.test_size,\n                \"full_train\": args.full_train,\n                \"initializer\": args.initializer,\n                \"architecture\": args.architecture,\n                \"activation\": args.activation,\n                \"block_dropout\": args.block_dropout,\n                \"head_hidden\": args.head_hidden,\n                \"head_dropout\": args.head_dropout,\n                \"time_reverse_probability\": args.time_reverse_probability,\n                \"contrast_strength\": args.contrast_strength,\n                \"optimizer\": args.optimizer,\n                \"scheduler\": args.scheduler,\n                \"lr_milestones\": lr_milestones,\n                \"plateau_patience\": args.plateau_patience,\n                \"plateau_factor\": args.plateau_factor,\n                \"min_lr\": args.min_lr,\n                \"early_stopping_patience\": args.early_stopping_patience,\n                \"early_stopped\": early_stopped,\n                \"submission_path\": str(submission_path),\n                \"model_path\": str(best_model_path),\n            },\n            indent=2,\n        )\n        + \"\\n\"\n    )\n    if args.full_train:\n        print(f\"full_train_epoch={checkpoint['epoch']}\")\n    else:\n        print(f\"best_lwlrap={best_lwlrap:.6f} epoch={checkpoint['epoch']}\")\n    print(f\"wrote {submission_path}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
}

len(EMBEDDED_FILES), sorted(EMBEDDED_FILES)


(16,
 ['__init__.py',
  'build_f1024_memmap_caches.py',
  'build_logmel_image_cache.py',
  'build_noisy_caches.py',
  'evaluate_and_blend.py',
  'fat2019/__init__.py',
  'fat2019/data.py',
  'fat2019/features.py',
  'fat2019/labels.py',
  'fat2019/metrics.py',
  'fat2019/neural_helpers.py',
  'fat2019/spectrogram_images.py',
  'fat2019/submission.py',
  'final_config.py',
  'finetune_mixed_noisy.py',
  'train_logmel_cnn.py'])

## Materializacion de fuentes embebidas

El notebook escribe esas fuentes a una carpeta de trabajo generada y
despues importa la configuracion final desde ahi. Si se copia solo
este notebook junto con los datos crudos, el codigo necesario viaja
dentro del propio `.ipynb`.


In [3]:
def materialize_embedded_source(*, force: bool = True) -> list[Path]:
    written: list[Path] = []
    for relative_path, source in EMBEDDED_FILES.items():
        destination = EMBEDDED_SRC / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        if force or not destination.exists() or destination.read_text(encoding="utf-8") != source:
            destination.write_text(source, encoding="utf-8")
        written.append(destination)
    return written


materialized_files = materialize_embedded_source()
if str(EMBEDDED_SRC) not in sys.path:
    sys.path.insert(0, str(EMBEDDED_SRC))

from final_config import BRANCHES, FINE_TUNE

{
    "embedded_files": len(EMBEDDED_FILES),
    "materialized_to": str(EMBEDDED_SRC.relative_to(ROOT)),
    "first_files": sorted(EMBEDDED_FILES)[:5],
}


{'embedded_files': 16,
 'materialized_to': 'proyecto_actual_v2/codigo/work/_embedded_pipeline_src',
 'first_files': ['__init__.py',
  'build_f1024_memmap_caches.py',
  'build_logmel_image_cache.py',
  'build_noisy_caches.py',
  'evaluate_and_blend.py']}

## Configuracion fija del modelo final


In [4]:
branch_config = pd.DataFrame(
    [
        {
            "branch": branch.name,
            "weight": branch.ensemble_weight,
            "architecture": branch.architecture,
            "activation": branch.activation,
            "frames": branch.frames,
            "normalization": "global-mel" if branch.cache_tag == "globalmel" else "clip",
            "base_epochs": branch.base_epochs,
            "batch_size": branch.batch_size,
            "optimizer": branch.optimizer,
            "scheduler": branch.scheduler,
            "lr_milestones": ",".join(map(str, branch.lr_milestones)),
            "head_hidden": branch.head_hidden,
            "head_dropout": branch.head_dropout,
        }
        for branch in BRANCHES
    ]
)

finetune_config = {
    "epochs_extra": FINE_TUNE.epochs,
    "learning_rate": FINE_TUNE.lr,
    "scheduler": "cosine",
    "min_lr": FINE_TUNE.min_lr,
    "batch_composition": "50% curated / 50% noisy",
    "curated_loss_weight": FINE_TUNE.curated_loss_weight,
    "noisy_loss_weight": FINE_TUNE.noisy_loss_weight,
    "gaussian_noise_std": FINE_TUNE.gaussian_noise_std,
    "augmentations": list(FINE_TUNE.augmentations),
    "ensemble_weights": [branch.ensemble_weight for branch in BRANCHES],
    "freesound2019_private_lb": FREESOUND2019_PRIVATE_LB,
    "course_public_lb": COURSE_PUBLIC_LB,
}

display(branch_config)
finetune_config


,branch,weight,architecture,activation,frames,normalization,base_epochs,batch_size,optimizer,scheduler,lr_milestones,head_hidden,head_dropout
0,separable_headsep,0.250,separable_residual,relu,512,clip,100,24,adam,multistep,"27,36,43,49,52",256,0.3
1,globalmel_sep_temporal,0.375,separable_temporal_bigru,silu,512,global-mel,100,24,adamw,multistep,"25,39",0,0.3
2,sep_temporal_f1024,0.375,separable_temporal_bigru,silu,1024,clip,100,12,adamw,multistep,"19,25",0,0.3


{'epochs_extra': 30,
 'learning_rate': 0.0001,
 'scheduler': 'cosine',
 'min_lr': 1e-06,
 'batch_composition': '50% curated / 50% noisy',
 'curated_loss_weight': 1.0,
 'noisy_loss_weight': 0.3,
 'gaussian_noise_std': 0.015,
 'augmentations': ['time shift',
  'time mask',
  'frequency mask',
  'gaussian noise'],
 'ensemble_weights': [0.25, 0.375, 0.375],
 'freesound2019_private_lb': 0.68122,
 'course_public_lb': 0.64307}

## Datos crudos y validaciones iniciales


In [5]:
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
curated = pd.read_csv(DATA_DIR / "train_curated.csv")
noisy = pd.read_csv(DATA_DIR / "train_noisy.csv")
label_columns = [column for column in sample.columns if column != "fname"]

raw_checks = {
    "sample_rows": len(sample),
    "curated_rows": len(curated),
    "noisy_rows": len(noisy),
    "num_labels": len(label_columns),
    "curated_unique_files": curated["fname"].nunique(),
    "noisy_unique_files": noisy["fname"].nunique(),
    "sample_has_fname": "fname" in sample.columns,
    "curated_has_labels": {"fname", "labels"}.issubset(curated.columns),
    "noisy_has_labels": {"fname", "labels"}.issubset(noisy.columns),
}

for split_name, frame in [("train_curated", curated), ("train_noisy", noisy)]:
    missing = [
        fname
        for fname in frame["fname"].astype(str).head(20)
        if not (DATA_DIR / fname).exists()
    ]
    if missing:
        raise FileNotFoundError(f"{split_name}: faltan archivos de muestra: {missing[:3]}")

missing_test = [
    fname for fname in sample["fname"].astype(str).head(20) if not (DATA_DIR / fname).exists()
]
if missing_test:
    raise FileNotFoundError(f"test: faltan archivos de muestra: {missing_test[:3]}")

assert raw_checks["num_labels"] == 80
assert raw_checks["sample_rows"] == 3361
assert raw_checks["curated_has_labels"]
assert raw_checks["noisy_has_labels"]
raw_checks


{'sample_rows': 3361,
 'curated_rows': 4970,
 'noisy_rows': 19815,
 'num_labels': 80,
 'curated_unique_files': 4970,
 'noisy_unique_files': 19815,
 'sample_has_fname': True,
 'curated_has_labels': True,
 'noisy_has_labels': True}

## Funciones de ejecucion y validacion


In [6]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def run_command(command: list[str], *, enabled: bool) -> None:
    print("$ " + " ".join(map(str, command)))
    if not enabled:
        print("skip: RUN_FULL_PIPELINE=False")
        return
    completed = subprocess.run(
        command,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"command failed with code {completed.returncode}")


@dataclass
class PipelineStep:
    name: str
    command: list[str]


class FinalAudioTaggingPipelineLargo:
    def __init__(self, *, data_dir: Path, code_dir: Path, work_dir: Path, embedded_src: Path) -> None:
        self.data_dir = data_dir
        self.code_dir = code_dir
        self.work_dir = work_dir
        self.embedded_src = embedded_src
        self.python = str(PYTHON_EXE)

    def cache_steps(self) -> list[PipelineStep]:
        base = [self.python, str(self.embedded_src / "build_logmel_image_cache.py"), "--data-dir", str(self.data_dir)]
        return [
            PipelineStep("logmel_512_clip_curated_test", base + ["--splits", "curated,test", "--n-mels", "128", "--frames", "512", "--normalization", "clip"]),
            PipelineStep("logmel_512_globalmel_curated_test", base + ["--splits", "curated,test", "--n-mels", "128", "--frames", "512", "--normalization", "global-mel", "--cache-tag", "globalmel"]),
            PipelineStep("logmel_1024_clip_curated_test", base + ["--splits", "curated,test", "--n-mels", "128", "--frames", "1024", "--normalization", "clip"]),
            PipelineStep("logmel_noisy_512", [self.python, str(self.embedded_src / "build_noisy_caches.py"), "--data-dir", str(self.data_dir)]),
            PipelineStep("logmel_1024_memmap_all", [self.python, str(self.embedded_src / "build_f1024_memmap_caches.py"), "--data-dir", str(self.data_dir)]),
        ]

    def base_train_steps(self) -> list[PipelineStep]:
        steps: list[PipelineStep] = []
        for branch in BRANCHES:
            command = [
                self.python,
                str(self.embedded_src / "train_logmel_cnn.py"),
                "--data-dir", str(self.data_dir),
                "--models-dir", str(self.work_dir / "models" / branch.source_run),
                "--submissions-dir", str(self.work_dir / "submissions" / branch.source_run),
                "--experiments-dir", str(self.work_dir / "experiments" / branch.source_run),
                "--n-mels", str(branch.n_mels),
                "--frames", str(branch.frames),
                "--epochs", str(branch.base_epochs),
                "--batch-size", str(branch.batch_size),
                "--lr", str(branch.base_lr),
                "--weight-decay", str(branch.base_weight_decay),
                "--optimizer", branch.optimizer,
                "--initializer", branch.initializer,
                "--architecture", branch.architecture,
                "--activation", branch.activation,
                "--head-hidden", str(branch.head_hidden),
                "--head-dropout", str(branch.head_dropout),
                "--block-dropout", str(branch.block_dropout),
                "--time-reverse-probability", str(branch.time_reverse_probability),
                "--contrast-strength", str(branch.contrast_strength),
                "--scheduler", branch.scheduler,
                "--lr-milestones", ",".join(map(str, branch.lr_milestones)),
                "--min-lr", str(FINE_TUNE.min_lr),
                "--seed", str(FINE_TUNE.seed),
                "--full-train",
            ]
            if branch.cache_tag is not None:
                command.extend(["--cache-tag", branch.cache_tag])
            steps.append(PipelineStep(f"train_curated_{branch.name}", command))
        return steps

    def finetune_steps(self) -> list[PipelineStep]:
        return [
            PipelineStep(
                "finetune_curated_noisy",
                [
                    self.python,
                    str(self.embedded_src / "finetune_mixed_noisy.py"),
                    "--data-dir", str(self.data_dir),
                    "--work-dir", str(self.work_dir),
                    "--branches", "all",
                    "--epochs", str(FINE_TUNE.epochs),
                    "--lr", str(FINE_TUNE.lr),
                    "--min-lr", str(FINE_TUNE.min_lr),
                    "--noisy-loss-weight", str(FINE_TUNE.noisy_loss_weight),
                    "--curated-loss-weight", str(FINE_TUNE.curated_loss_weight),
                    "--gaussian-noise-std", str(FINE_TUNE.gaussian_noise_std),
                    "--seed", str(FINE_TUNE.seed),
                    "--num-workers", str(FINE_TUNE.num_workers),
                    "--submission-checkpoint", FINE_TUNE.submission_checkpoint,
                ],
            ),
            PipelineStep(
                "blend_final_submission",
                [
                    self.python,
                    str(self.embedded_src / "evaluate_and_blend.py"),
                    "--data-dir", str(self.data_dir),
                    "--work-dir", str(self.work_dir),
                    "--output-dir", str(self.code_dir),
                ],
            ),
        ]

    def steps(self) -> list[PipelineStep]:
        return self.cache_steps() + self.base_train_steps() + self.finetune_steps()

    def run(self, *, enabled: bool) -> pd.DataFrame:
        rows = []
        for step in self.steps():
            run_command(step.command, enabled=enabled)
            rows.append({"step": step.name, "command": " ".join(map(str, step.command))})
        return pd.DataFrame(rows)

    def validate_submission(self, path: Path) -> dict[str, object]:
        if not path.exists():
            raise FileNotFoundError(f"No existe {path}. Ejecutar el pipeline completo para generarlo.")
        submission = pd.read_csv(path)
        assert submission.shape == (len(sample), len(sample.columns))
        assert list(submission.columns) == list(sample.columns)
        assert submission["fname"].astype(str).equals(sample["fname"].astype(str))
        values = submission[label_columns].to_numpy(dtype=float)
        assert np.isfinite(values).all()
        assert values.min() >= 0.0
        assert values.max() <= 1.0
        return {
            "path": str(path.relative_to(ROOT)),
            "rows": len(submission),
            "labels": len(label_columns),
            "min_probability": float(values.min()),
            "max_probability": float(values.max()),
            "sha256": sha256(path),
        }


pipeline = FinalAudioTaggingPipelineLargo(
    data_dir=DATA_DIR,
    code_dir=CODE_DIR,
    work_dir=WORK_DIR,
    embedded_src=EMBEDDED_SRC,
)
planned_steps = pd.DataFrame(
    [{"step": step.name, "command": " ".join(map(str, step.command))} for step in pipeline.steps()]
)
display(planned_steps)


,step,command
0,logmel_512_clip_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
1,logmel_512_globalmel_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
2,logmel_1024_clip_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
3,logmel_noisy_512,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
4,logmel_1024_memmap_all,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
5,train_curated_separable_headsep,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
6,train_curated_globalmel_sep_temporal,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
7,train_curated_sep_temporal_f1024,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
8,finetune_curated_noisy,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
9,blend_final_submission,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...


## Ejecucion del pipeline


In [7]:
RUN_FULL_PIPELINE = False

executed_steps = pipeline.run(enabled=RUN_FULL_PIPELINE)
submission_checks = pipeline.validate_submission(CODE_DIR / "submission.csv")
assert submission_checks["sha256"] == EXPECTED_FINAL_SHA256

display(executed_steps)
submission_checks


$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/proyecto_actual_v2/codigo/work/_embedded_pipeline_src/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --splits curated,test --n-mels 128 --frames 512 --normalization clip
skip: RUN_FULL_PIPELINE=False
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/proyecto_actual_v2/codigo/work/_embedded_pipeline_src/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --splits curated,test --n-mels 128 --frames 512 --normalization global-mel --cache-tag globalmel
skip: RUN_FULL_PIPELINE=False
$ /home/santig14/fing/taa/2_TallerAA/.venv/bin/python /home/santig14/fing/taa/2_TallerAA/proyecto_actual_v2/codigo/work/_embedded_pipeline_src/build_logmel_image_cache.py --data-dir /home/santig14/fing/taa/2_TallerAA/data --splits curated,test --n-mels 128 --frames 1024 --normalization clip
skip: RUN_FULL_PIPELINE=False
$ /home/

,step,command
0,logmel_512_clip_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
1,logmel_512_globalmel_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
2,logmel_1024_clip_curated_test,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
3,logmel_noisy_512,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
4,logmel_1024_memmap_all,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
5,train_curated_separable_headsep,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
6,train_curated_globalmel_sep_temporal,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
7,train_curated_sep_temporal_f1024,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
8,finetune_curated_noisy,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...
9,blend_final_submission,/home/santig14/fing/taa/2_TallerAA/.venv/bin/p...


{'path': 'proyecto_actual_v2/codigo/submission.csv',
 'rows': 3361,
 'labels': 80,
 'min_probability': 1.8894305625e-10,
 'max_probability': 0.99999625,
 'sha256': 'b29288caa6e7b37b29e830a29655decc7c6bc8110ca3a40b828f4dd2f5fabdcc'}

## Figuras y metadata final


In [8]:
plt.style.use("default")

split_counts = pd.DataFrame(
    [
        {"split": "train_curated", "rows": len(curated)},
        {"split": "train_noisy", "rows": len(noisy)},
        {"split": "test", "rows": len(sample)},
    ]
)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(split_counts["split"], split_counts["rows"], color=["#2f6f73", "#d28b35", "#575a89"])
ax.set_title("Tamanos de splits")
ax.set_ylabel("audios")
ax.grid(axis="y", alpha=0.25)
for index, value in enumerate(split_counts["rows"]):
    ax.text(index, value + 300, f"{value}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_v2_largo_split_sizes.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.2, 3.5))
ax.pie(
    branch_config["weight"],
    labels=branch_config["branch"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4b7f52", "#386c9f", "#ba7a2a"],
)
ax.set_title("Pesos del ensamble final")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_v2_largo_component_weights.png", dpi=160)
plt.close(fig)

figures = sorted(path.name for path in FIGURES_DIR.glob("fig_v2_largo_*.png"))

metadata = {
    "pipeline": {
        "self_contained_from_raw_data": True,
        "contains_embedded_source": True,
        "uses_pipeline_src_as_runtime_dependency": False,
        "uses_investigation_artifacts_as_source": False,
        "run_full_pipeline_flag": RUN_FULL_PIPELINE,
        "embedded_source_dir": str(EMBEDDED_SRC.relative_to(ROOT)),
        "embedded_file_count": len(EMBEDDED_FILES),
    },
    "raw_data_checks": raw_checks,
    "branch_config": branch_config.to_dict(orient="records"),
    "finetune_config": finetune_config,
    "submission": submission_checks,
    "leaderboard": {
        "previous_3way_private_lb": PREVIOUS_3WAY_PRIVATE_LB,
        "freesound2019_private_lb": FREESOUND2019_PRIVATE_LB,
        "course_public_lb": COURSE_PUBLIC_LB,
    },
    "planned_steps": planned_steps.to_dict(orient="records"),
    "figures": figures,
}
metadata_path = CODE_DIR / "pipeline_final_taller_2_v2_largo_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False) + "\n")
metadata


{'pipeline': {'self_contained_from_raw_data': True,
  'contains_embedded_source': True,
  'uses_pipeline_src_as_runtime_dependency': False,
  'uses_investigation_artifacts_as_source': False,
  'run_full_pipeline_flag': False,
  'embedded_source_dir': 'proyecto_actual_v2/codigo/work/_embedded_pipeline_src',
  'embedded_file_count': 16},
 'raw_data_checks': {'sample_rows': 3361,
  'curated_rows': 4970,
  'noisy_rows': 19815,
  'num_labels': 80,
  'curated_unique_files': 4970,
  'noisy_unique_files': 19815,
  'sample_has_fname': True,
  'curated_has_labels': True,
  'noisy_has_labels': True},
 'branch_config': [{'branch': 'separable_headsep',
   'weight': 0.25,
   'architecture': 'separable_residual',
   'activation': 'relu',
   'frames': 512,
   'normalization': 'clip',
   'base_epochs': 100,
   'batch_size': 24,
   'optimizer': 'adam',
   'scheduler': 'multistep',
   'lr_milestones': '27,36,43,49,52',
   'head_hidden': 256,
   'head_dropout': 0.3},
  {'branch': 'globalmel_sep_temporal',